<a href="https://colab.research.google.com/github/Kazi-Sadman/Drug-solubility-prediction/blob/main/Thesis_new_work_organized.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# An Explainable Language-Aware Cross-Lingual Framework for Bangla–English Fake News Detection

**Multilingual Transformers (mBERT, XLM-R, optional LaBSE/BanglaBERT) + Classical ML (SVM, LR, NB, XGBoost) + LIME**

How to use this notebook (Google Colab):
1. `Runtime → Change runtime type → T4 GPU`.
2. Run the cells **top to bottom**. Each section has a text cell (what it does) followed by its code cell.
3. When Section 4 asks for the dataset, upload `ben_fnd_16k.csv` (or keep it in `/content` or Google Drive).
4. The **last section** zips every output folder (figures, reports, tables, LIME, predictions, …) and downloads it to your PC.

| Part | Sections | Thesis objectives |
|---|---|---|
| A. Setup | 1–3 | reproducibility |
| B. Data | 4–8 | dataset prep, source documentation, leakage check (obj. 4) |
| C. Classical ML | 9–11 | obj. 1, 2 (tuned SVM / LR / NB + XGBoost, multi-seed) |
| D. Transformers | 12–14 | obj. 3, 5, 6 (mBERT, XLM-R, LaBSE; Bn-only, En-only, joint) |
| E. Evaluation | 15–17 | obj. 6, 7 (A–E experiments, CI, confusion matrices) + few-shot + leave-one-category-out |
| F. Selection | 18 | obj. 8 (validation-based, confidence router) |
| G. Explainability and errors | 19–20 | obj. 9, 10 |
| H. Final system | 21–23 | obj. 11, 12 (LIME system, Gradio demo, master table) |
| I. Export | 24–25 | save artifacts, download everything to your PC |

**Run modes.** Set `RUN_MODE = "smoke"` in Section 2 for a ~15-minute dry run on 1,500 rows (checks that nothing crashes). Use `"full"` for thesis results.

## SECTION 1 — ENVIRONMENT SETUP

**What we do:** Check the GPU, install all libraries once.

**Output:** GPU name/memory printed; packages installed.

**সংক্ষেপে:** GPU ঠিকমতো আছে কিনা দেখা হয় এবং দরকারি লাইব্রেরি ইনস্টল হয়।

> ⚙️ **Colab note:** Safe to re-run. If you see `CPU only`, change the runtime to T4 GPU — transformer training would take many hours on CPU.

In [ ]:
# =============================================================================
# SECTION 1 — ENVIRONMENT SETUP
# What we do : verify GPU, install dependencies (pip skips what is already installed)
# =============================================================================
try:
    import torch
    if torch.cuda.is_available():
        print("✅ GPU CONNECTED:", torch.cuda.get_device_name(0),
              "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
    else:
        print("❌ CPU only — go to Runtime → Change runtime type → T4 GPU")
except ImportError:
    print("torch not installed (fine for classical-only dry runs)")

!pip install -q pandas numpy scikit-learn xgboost matplotlib seaborn scipy \
    transformers datasets accelerate sentencepiece langdetect lime gradio \
    joblib tqdm openpyxl

## SECTION 2 — IMPORTS, CONFIGURATION AND SEEDS

**What we do:** Import libraries, define **every** setting in one place (run mode, seeds, model names, T4 speed settings, output folders).

**Output:** A `thesis_outputs/` folder tree (figures, reports, tables, predictions, lime, models) and `thesis_work/` (checkpoints, caches).

**সংক্ষেপে:** সব সেটিং একজায়গায়। শুধু এই সেকশন বদলালেই পুরো পাইপলাইনের আচরণ বদলায়।

> ⚙️ **Colab note:** Set `USE_GOOGLE_DRIVE = True` so trained models and cached predictions survive a Colab disconnect (strongly recommended for the full run).

In [ ]:
# =============================================================================
# SECTION 2 — IMPORTS, CONFIGURATION, SEEDS
# What we do : one place for all settings; every later section only reads them.
# =============================================================================
import os, re, io, sys, json, time, html, glob, random, shutil, pickle, inspect, platform
import unicodedata, warnings, zipfile
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix)
from sklearn.metrics.pairwise import cosine_similarity
try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# ------------------------- RUN MODE -----------------------------------------
RUN_MODE            = "full"     # "smoke" = quick dry run (1,500 rows) | "full" = thesis run
USE_GOOGLE_DRIVE    = False      # True -> checkpoints/caches saved in Drive (survive disconnects)
USE_CLEANED_TEXT    = True       # True -> remove leakage artifacts (Section 8) before modelling

# ------------------------- DATA SCHEMA --------------------------------------
COL_ID, COL_TITLE, COL_BODY = "id", "title", "news_details"
COL_LANG, COL_CAT, COL_LABEL = "language", "category", "label"
LABEL_MAP   = {"Real": 0, "Fake": 1}
LABEL_NAMES = ["Real", "Fake"]
LANGS       = ["bangla", "english"]
CODE_MIXED_MIN_LATIN_CHARS = 10   # Bangla row with >=10 Latin letters counts as code-mixed
TEXT_COL    = "model_text"        # column every model reads (set in Section 8)

# ------------------------- SPLIT / TF-IDF / CLASSICAL -----------------------
SEED = 42
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15
NEAR_DUP_COSINE = 0.90
TOKEN_PATTERN = r"[\u0980-\u09FF]+|[A-Za-z0-9]+"   # FIX: keeps Bangla words (incl. vowel signs) intact
CV_FOLDS = 5
N_SEEDS_CLASSICAL = 5             # repeated random splits for mean +/- std (classical models)
MULTISEED_INCLUDE_XGB = False
RUN_CLASSICAL_MULTISEED = True

# ------------------------- TRANSFORMERS (T4-friendly) -----------------------
TRANSFORMERS = {"mBERT": "bert-base-multilingual-cased", "XLM-R": "xlm-roberta-base"}
FINE_TUNE_LABSE   = False         # add LaBSE as a fine-tuned classifier too (slower)
RUN_BANGLABERT    = False         # add BanglaBERT (Bangla-only experiments)
BANGLABERT_NAME   = "csebuetnlp/banglabert"
LABSE_NAME        = "setu4993/LaBSE"
MAX_LEN           = 256           # keep 256; try 512 only for the very last run (~3x slower)
BATCH_SIZE        = 16
GRAD_ACCUM        = 2             # effective batch = 32
LEARNING_RATE     = 2e-5
WEIGHT_DECAY      = 0.01
WARMUP_RATIO      = 0.10
TRANSFORMER_EPOCHS = 3            # set 2 for a faster, still safe run (early stopping patience = 1)
EARLY_STOP_PATIENCE = 1
N_TRANSFORMER_SEEDS = 1           # >1 -> extra seeds for XLM-R joint (Section 15b), 3 recommended for final
REUSE_SAVED_MODELS  = True        # skip training if a model folder already exists

# model registry: which models are trained in which scopes (scope = training data: joint / bangla / english)
TRANSFORMER_MODELS = dict(TRANSFORMERS)
if FINE_TUNE_LABSE: TRANSFORMER_MODELS["LaBSE"] = LABSE_NAME
if RUN_BANGLABERT:  TRANSFORMER_MODELS["BanglaBERT"] = BANGLABERT_NAME
MODEL_SCOPES = {m: ["joint", "bangla", "english"] for m in TRANSFORMER_MODELS}
if RUN_BANGLABERT: MODEL_SCOPES["BanglaBERT"] = ["bangla"]       # Bangla-only pre-trained model

try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False
DEVICE = "cuda" if HAS_GPU else "cpu"

# ------------------------- EXPERIMENT EXTRAS --------------------------------
FEWSHOT_SIZES   = [0, 50, 100, 500]   # target-language examples for the few-shot curve
FEWSHOT_MODEL   = "XLM-R"
FEWSHOT_EPOCHS  = 3
N_BOOTSTRAP     = 1000
LOCO_MIN_ROWS, LOCO_MIN_PER_CLASS = 100, 20
LIME_EXAMPLES_PER_LANGUAGE = 5
LIME_SAMPLES_CLASSICAL, LIME_SAMPLES_TRANSFORMER = 1000, 300
ERRORS_TO_TAG   = 100
EXPORT_MODEL_WEIGHTS = False      # True -> weights (GBs!) are also zipped into the download

if RUN_MODE == "smoke":
    SMOKE_ROWS, TRANSFORMER_EPOCHS, N_BOOTSTRAP, CV_FOLDS = 1500, 1, 200, 3
    N_SEEDS_CLASSICAL, FEWSHOT_SIZES, FEWSHOT_EPOCHS = 2, [0, 20, 50], 1
    LIME_EXAMPLES_PER_LANGUAGE, LIME_SAMPLES_CLASSICAL, LIME_SAMPLES_TRANSFORMER = 2, 200, 100
    LOCO_MIN_ROWS, LOCO_MIN_PER_CLASS, ERRORS_TO_TAG = 40, 5, 20
else:
    SMOKE_ROWS = None

# ------------------------- FOLDERS ------------------------------------------
BASE_DIR    = "/content" if os.path.isdir("/content") else "."
OUTPUT_ROOT = f"{BASE_DIR}/thesis_outputs"
WORK_DIR    = f"{BASE_DIR}/thesis_work"
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/thesis_work"
FIGURES_DIR, CM_DIR = f"{OUTPUT_ROOT}/figures", f"{OUTPUT_ROOT}/figures/confusion_matrices"
REPORTS_DIR, TABLES_DIR = f"{OUTPUT_ROOT}/reports", f"{OUTPUT_ROOT}/tables"
PRED_DIR, LIME_DIR, MODELS_DIR = f"{OUTPUT_ROOT}/predictions", f"{OUTPUT_ROOT}/lime", f"{OUTPUT_ROOT}/models"
for d in [FIGURES_DIR, CM_DIR, REPORTS_DIR, TABLES_DIR, PRED_DIR, LIME_DIR, MODELS_DIR,
          f"{WORK_DIR}/models", f"{WORK_DIR}/ckpt", f"{WORK_DIR}/cache"]:
    Path(d).mkdir(parents=True, exist_ok=True)

# ------------------------- SEEDS --------------------------------------------
def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
set_all_seeds(SEED)
print(f"RUN_MODE={RUN_MODE} | outputs -> {OUTPUT_ROOT} | work -> {WORK_DIR}")

## SECTION 3 — HELPER FUNCTIONS

**What we do:** Small reusable tools: JSON/figure saving, metrics, fast bootstrap confidence intervals, confusion-matrix heatmap.

**Output:** No output (functions only).

**সংক্ষেপে:** পরে বারবার লাগবে এমন ছোট ফাংশন (মেট্রিক, CI, heatmap, সেভ)।

In [ ]:
# =============================================================================
# SECTION 3 — HELPER FUNCTIONS
# What we do : metrics, bootstrap CI, heatmap, saving utilities (used everywhere below)
# =============================================================================
def banner(n, title):
    print("\n" + "=" * 78 + f"\nSECTION {n} — {title}\n" + "=" * 78)

def _jsonable(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray):     return o.tolist()
    return str(o)

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_jsonable)

def save_fig(name, folder=None, show=True):
    folder = folder or FIGURES_DIR
    plt.savefig(f"{folder}/{name}.png", dpi=150, bbox_inches="tight")
    if show: plt.show()
    plt.close()

def compute_metrics(y_true, prob_fake, threshold=0.5):
    """Accuracy, macro P/R/F1, weighted F1, per-class F1, ROC-AUC, confusion matrix."""
    y_true = np.asarray(y_true); pred = (np.asarray(prob_fake) >= threshold).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y_true, pred, average="macro", zero_division=0)
    out = {"n": int(len(y_true)), "accuracy": accuracy_score(y_true, pred),
           "precision_macro": p, "recall_macro": r, "f1_macro": f,
           "f1_weighted": f1_score(y_true, pred, average="weighted", zero_division=0),
           "f1_fake": f1_score(y_true, pred, pos_label=1, zero_division=0),
           "f1_real": f1_score(y_true, pred, pos_label=0, zero_division=0),
           "roc_auc": roc_auc_score(y_true, prob_fake) if len(np.unique(y_true)) == 2 else np.nan,
           "cm": confusion_matrix(y_true, pred, labels=[0, 1])}
    return out

def bootstrap_macro_f1_ci(y_true, prob_fake, n_boot=None, seed=SEED, threshold=0.5):
    """95% bootstrap CI of macro-F1 (vectorised, ~milliseconds)."""
    n_boot = n_boot or N_BOOTSTRAP
    y = np.asarray(y_true); p = (np.asarray(prob_fake) >= threshold).astype(int)
    rng = np.random.default_rng(seed); idx = rng.integers(0, len(y), (n_boot, len(y)))
    yb, pb = y[idx], p[idx]; f1s = []
    for c in (0, 1):
        tp = ((yb == c) & (pb == c)).sum(1); fp = ((yb != c) & (pb == c)).sum(1); fn = ((yb == c) & (pb != c)).sum(1)
        f1s.append(2 * tp / np.maximum(2 * tp + fp + fn, 1))
    m = np.mean(f1s, axis=0)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

def plot_confusion(ax, cm, title):
    """Heatmap with counts and row-percentages."""
    cm = np.asarray(cm); pct = cm / np.maximum(cm.sum(1, keepdims=True), 1) * 100
    annot = np.array([[f"{cm[i, j]}\n({pct[i, j]:.1f}%)" for j in range(2)] for i in range(2)])
    sns.heatmap(cm, annot=annot, fmt="", cmap="Blues", cbar=False, ax=ax,
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title, fontsize=9)

def script_counts(text):
    t = str(text)
    return len(re.findall(r"[\u0980-\u09FF]", t)), len(re.findall(r"[A-Za-z]", t))

def detect_script_language(text):
    """'bangla' | 'english' by dominant script (used for routing and for filling missing labels)."""
    bn, la = script_counts(text)
    return "bangla" if bn > la else "english"
print("helpers ready")

## SECTION 4 — DATASET LOADING AND DOCUMENTATION

**What we do:** Find/upload the CSV, load it, and create a **dataset card** (source documentation required by objective 4).

**Output:** `reports/dataset_card.json`. Fill the `TODO` fields with your real sources (names, URLs, years, licence).

**সংক্ষেপে:** ডেটাসেট লোড হয় এবং ডেটার সোর্স ডকুমেন্টেশনের জন্য একটি কার্ড তৈরি হয় — TODO অংশগুলো নিজে পূরণ করো।

> ⚙️ **Colab note:** If the file is not found, a file-upload button appears. Upload `ben_fnd_16k.csv` (a name like `ben_fnd_16k (1).csv` also works).

In [ ]:
# =============================================================================
# SECTION 4 — DATASET LOADING AND DOCUMENTATION
# What we do : locate the CSV (cwd, /content, Drive, or upload), load it, write the dataset card
# =============================================================================
banner(4, "DATASET LOADING")

def find_dataset():
    for pattern in [f"{BASE_DIR}/ben_fnd_16k*.csv", "./ben_fnd_16k*.csv", f"{BASE_DIR}/*.csv",
                    "/content/drive/MyDrive/**/ben_fnd_16k*.csv"]:
        hits = sorted(glob.glob(pattern, recursive=True))
        if hits:
            return hits[0]
    try:
        from google.colab import files
        print("Upload ben_fnd_16k.csv:")
        return list(files.upload().keys())[0]
    except ImportError:
        raise FileNotFoundError("Put ben_fnd_16k.csv next to the notebook.")

RAW_PATH = find_dataset()
raw_df = pd.read_csv(RAW_PATH, engine="python", on_bad_lines="skip")
raw_df.columns = [c.strip() for c in raw_df.columns]
print("File:", RAW_PATH, "| shape:", raw_df.shape, "| columns:", list(raw_df.columns))

# ---- Dataset card: FILL THE TODO FIELDS (examiners will ask where the data comes from) ----
DATASET_CARD = {
    "file": os.path.basename(RAW_PATH),
    "rows_raw": int(len(raw_df)),
    "columns": list(raw_df.columns),
    "english_source": "TODO: name, URL, collection period, licence (the English part looks like a public "
                      "Kaggle-style news corpus with categories such as politicsNews / worldnews / News — please confirm and cite)",
    "bangla_source":  "TODO: name, URL, collection period, licence (how were Bangla Fake/Real labels assigned?)",
    "labelling_method": "TODO: who labelled, inter-annotator agreement if any",
    "known_issues": ["category column is almost label-pure in English (leakage proxy) — never used as a feature",
                     "English text contains source artifacts (see Section 8)"],
}
raw_df.head(3)

## SECTION 5 — CLEANING, AUDIT, LANGUAGE VERIFICATION AND CODE-MIXED FLAGS

**What we do:** Drop NaN rows, normalise text (HTML, Unicode NFC, Bangla digits, zero-width chars …), build `title + body` text, verify the language label by script, flag **code-mixed** rows (Bangla rows containing English words), and create two text versions: `text_raw` and `text_clean` (leakage artifacts removed).

**Output:** A clean DataFrame `df`, `reports/dataset_audit.json`.

**সংক্ষেপে:** NaN রো বাদ, টেক্সট নরমালাইজ, ভাষার লেবেল যাচাই, code-mixed রো চিহ্নিত।

In [ ]:
# =============================================================================
# SECTION 5 — CLEANING, AUDIT, LANGUAGE VERIFICATION, CODE-MIXED FLAGS
# What we do : NaN handling, text normalisation, language check, code-mixed flag, leakage-artifact patterns
# =============================================================================
banner(5, "CLEANING AND AUDIT")

URL_RE    = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE  = re.compile(r"\S+@\S+\.\S+")
TAG_RE    = re.compile(r"<[^>]+>")
CTRL_RE   = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")
ZW_RE     = re.compile(r"[\u200B-\u200F\u202A-\u202E\u2060\uFEFF]")
WS_RE     = re.compile(r"\s+")
REP_LETTER_RE = re.compile(r"([^\W\d_])\1{2,}")      # soooo -> soo  (digits untouched: 1000 stays 1000)
REP_PUNCT_RE  = re.compile(r"([!?.])\1{2,}")
BN_DIGITS = {ord(b): ord("0") + i for i, b in enumerate("০১২৩৪৫৬৭৮৯")}
TYPO_MAP = {ord(a): b for a, b in zip("\u2018\u2019\u201C\u201D\u2013\u2014\u2026", ["'", "'", '"', '"', "-", "-", "..."])}

def normalize_text(t):
    """Script-agnostic normalisation. Keeps case and Bangla characters."""
    if not isinstance(t, str): return ""
    t = html.unescape(t); t = unicodedata.normalize("NFC", t)
    t = t.translate(TYPO_MAP)                      # curly quotes / dashes -> plain (typography can leak the source)
    t = TAG_RE.sub(" ", t); t = URL_RE.sub(" ", t); t = EMAIL_RE.sub(" ", t)
    t = t.translate(BN_DIGITS); t = ZW_RE.sub("", t); t = CTRL_RE.sub(" ", t)
    t = REP_LETTER_RE.sub(r"\1\1", t); t = REP_PUNCT_RE.sub(r"\1\1", t)
    return WS_RE.sub(" ", t).strip()

# Source artifacts that give the label away (found in the leakage audit; edit this list if you find more).
LEAKAGE_PATTERNS = {
    "reuters_tag":    r"\(Reuters\)|\bReuters\b",
    "featured_image": r"Featured image(?: via [^.]{0,60})?",
    "getty_images":   r"Getty Images",
    "pic_twitter":    r"pic\.twitter\.com/\S*",
    "video_tag":      r"\bVIDEO\b:?",
    "image_credit":   r"\b(?:Image|Photo|Screenshot)(?: via| by| credit| courtesy)[^.]{0,60}",
}
_LEAK_RES = {k: re.compile(v) for k, v in LEAKAGE_PATTERNS.items()}
def clean_artifacts(t):
    for rx in _LEAK_RES.values():
        t = rx.sub(" ", t)
    return WS_RE.sub(" ", t).strip()

def prepare_text(t):
    """Exactly what the models see: normalisation (+ artifact removal if USE_CLEANED_TEXT). Used by LIME and the demo."""
    t = normalize_text(t)
    return clean_artifacts(t) if USE_CLEANED_TEXT else t

# ---------- 5.1 Missing values ----------
df = raw_df.copy()
audit = {"rows_raw": int(len(df)), "missing_per_column": df.isna().sum().to_dict()}
df[COL_LANG] = df[COL_LANG].astype("object").where(df[COL_LANG].notna(), None)
df[COL_LANG] = df[COL_LANG].map(lambda x: x.strip().lower() if isinstance(x, str) else x)
df[COL_LANG] = df[COL_LANG].replace({"bn": "bangla", "en": "english"})

before = len(df)
df = df.dropna(subset=[COL_TITLE, COL_BODY, COL_LABEL]).copy()       # NaN title/body/label rows are dropped
audit["rows_dropped_nan"] = before - len(df)
missing_lang = df[COL_LANG].isna()
df.loc[missing_lang, COL_LANG] = [detect_script_language(t) for t in df.loc[missing_lang, COL_BODY]]
audit["language_recovered_from_script"] = int(missing_lang.sum())

df[COL_LABEL] = df[COL_LABEL].astype(str).str.strip().str.capitalize()
df = df[df[COL_LABEL].isin(LABEL_MAP)].copy()
df["label_id"] = df[COL_LABEL].map(LABEL_MAP).astype(int)

# ---------- 5.2 Text preparation ----------
df["title_norm"] = df[COL_TITLE].map(normalize_text)
df["body_norm"]  = df[COL_BODY].map(normalize_text)
df["text_raw"]   = (df["title_norm"] + ". " + df["body_norm"]).str.strip(". ").str.strip()
df["text_clean"] = df["text_raw"].map(clean_artifacts)
before = len(df); df = df[df["text_clean"].str.len() >= 20].copy()
audit["rows_dropped_too_short"] = before - len(df)

# ---------- 5.3 Language verification + code-mixed flag ----------
sc = df["text_raw"].map(script_counts)
df["n_bangla_chars"] = [a for a, b in sc]; df["n_latin_chars"] = [b for a, b in sc]
df["lang_script"] = np.where(df["n_bangla_chars"] > df["n_latin_chars"], "bangla", "english")
audit["language_label_vs_script"] = pd.crosstab(df[COL_LANG], df["lang_script"]).to_dict()
df["is_code_mixed"] = (((df[COL_LANG] == "bangla") & (df["n_latin_chars"] >= CODE_MIXED_MIN_LATIN_CHARS)) |
                       ((df[COL_LANG] == "english") & (df["n_bangla_chars"] >= CODE_MIXED_MIN_LATIN_CHARS)))
audit["code_mixed_rows"] = int(df["is_code_mixed"].sum())
df = df.reset_index(drop=True)

# ---------- 5.4 Smoke mode: small stratified sample ----------
if SMOKE_ROWS:
    df = (df.groupby([COL_LANG, "label_id"], group_keys=False)
            .sample(frac=min(1.0, SMOKE_ROWS / len(df)), random_state=SEED).reset_index(drop=True))
    print(f"[SMOKE MODE] using {len(df)} rows")

# ---------- 5.5 Audit summary ----------
audit.update({
    "rows_final": int(len(df)),
    "label_distribution": df[COL_LABEL].value_counts().to_dict(),
    "language_distribution": df[COL_LANG].value_counts().to_dict(),
    "label_by_language": pd.crosstab(df[COL_LANG], df[COL_LABEL]).to_dict(),
    "median_chars_by_language": df.groupby(COL_LANG)["text_raw"].apply(lambda s: float(s.str.len().median())).to_dict(),
    "code_mixed_by_language": df.groupby(COL_LANG)["is_code_mixed"].sum().to_dict(),
})
save_json(audit, f"{REPORTS_DIR}/dataset_audit.json")
DATASET_CARD["rows_after_cleaning"] = int(len(df)); DATASET_CARD["audit"] = audit
save_json(DATASET_CARD, f"{REPORTS_DIR}/dataset_card.json")
print(json.dumps({k: v for k, v in audit.items() if k != "missing_per_column"}, indent=2, ensure_ascii=False, default=_jsonable))

## SECTION 6 — EXPLORATORY DATA ANALYSIS

**What we do:** Plots: class/language balance, text length, and — importantly — **label by category** (shows which categories are label-pure and therefore a leakage risk).

**Output:** `figures/eda_overview.png`, `figures/category_vs_label.png`.

**সংক্ষেপে:** ডেটার ভারসাম্য, দৈর্ঘ্য এবং category-অনুযায়ী Fake/Real বণ্টন (লিকেজ বোঝার জন্য)।

In [ ]:
# =============================================================================
# SECTION 6 — EXPLORATORY DATA ANALYSIS
# What we do : overview plots + category-vs-label chart (leakage evidence)
# =============================================================================
banner(6, "EDA")
df["text_len"] = df["text_raw"].str.len()
fig, ax = plt.subplots(2, 3, figsize=(16, 8))
df[COL_LABEL].value_counts().plot.bar(ax=ax[0, 0], color=["#4c72b0", "#c44e52"]); ax[0, 0].set_title("Class distribution")
df[COL_LANG].value_counts().plot.bar(ax=ax[0, 1], color=["#55a868", "#dd8452"]); ax[0, 1].set_title("Language distribution")
sns.countplot(data=df, x=COL_LANG, hue=COL_LABEL, ax=ax[0, 2]); ax[0, 2].set_title("Fake/Real by language")
sns.boxplot(data=df, x=COL_LANG, y="text_len", ax=ax[1, 0]); ax[1, 0].set_yscale("log"); ax[1, 0].set_title("Text length by language (log)")
sns.boxplot(data=df, x=COL_LABEL, y="text_len", hue=COL_LANG, ax=ax[1, 1]); ax[1, 1].set_yscale("log"); ax[1, 1].set_title("Text length by class (log)")
df.groupby(COL_LANG)["is_code_mixed"].sum().plot.bar(ax=ax[1, 2], color="#8172b2"); ax[1, 2].set_title("Code-mixed rows")
plt.tight_layout(); save_fig("eda_overview")

# Category vs label: bars that are one colour = label-pure category = leakage proxy
cat_tab = pd.crosstab([df[COL_LANG], df[COL_CAT]], df[COL_LABEL])
cat_tab.plot.barh(stacked=True, figsize=(9, max(5, 0.28 * len(cat_tab))), color=["#4c72b0", "#c44e52"])
plt.title("Label counts per (language, category) — one-colour bars are label-pure"); plt.tight_layout(); save_fig("category_vs_label")
cat_tab.to_csv(f"{TABLES_DIR}/category_vs_label.csv")
pure = cat_tab[(cat_tab.min(axis=1) == 0)]
print(f"Label-pure (language, category) groups: {len(pure)} covering {int(pure.values.sum())} rows "
      f"({pure.values.sum() / len(df):.0%} of the data) -> category must NOT be used as a feature.")

## SECTION 7 — NEAR-DUPLICATE REMOVAL AND STRATIFIED SPLIT

**What we do:** Remove near-duplicate articles (TF-IDF cosine ≥ 0.90) so train/test cannot share the same story, then make a **70/15/15 split stratified by language × label**. The test set is frozen: it is only used for final reporting, never for choosing models.

**Output:** `predictions/split_assignment.csv`, `reports/split_report.json`.

**সংক্ষেপে:** ডুপ্লিকেট সরানো হয় এবং ভাষা×লেবেল অনুযায়ী 70/15/15 split হয়। টেস্ট সেট কেবল শেষ রিপোর্টে ব্যবহার হবে।

In [ ]:
# =============================================================================
# SECTION 7 — NEAR-DUPLICATE REMOVAL + STRATIFIED SPLIT
# What we do : union-find clusters of near-duplicates, keep one per cluster, 70/15/15 stratified split
# =============================================================================
import hashlib
banner(7, "DEDUPLICATION AND SPLIT")

def remove_near_duplicates(frame, text_col, threshold=NEAR_DUP_COSINE, chunk=1000):
    vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2), min_df=2,
                          max_features=50000, sublinear_tf=True, dtype=np.float32)
    X = vec.fit_transform(frame[text_col]); n = X.shape[0]; parent = list(range(n))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for s in range(0, n, chunk):
        sim = cosine_similarity(X[s:s + chunk], X)
        rows, cols = np.where(sim >= threshold)
        for r, c in zip(rows, cols):
            g = s + r
            if c > g:
                ra, rb = find(g), find(c)
                if ra != rb: parent[max(ra, rb)] = min(ra, rb)
    roots = np.array([find(i) for i in range(n)])
    clusters = pd.Series(roots).value_counts(); multi = clusters[clusters > 1].index
    conflicts = int(sum(frame["label_id"].values[roots == r].min() != frame["label_id"].values[roots == r].max() for r in multi))
    keep = roots == np.arange(n)
    return frame[keep].reset_index(drop=True), {"removed": int((~keep).sum()), "clusters": int(len(multi)), "clusters_with_label_conflict": conflicts}

df, dup_report = remove_near_duplicates(df, "text_clean")
print("Near-duplicate report:", dup_report)

def make_splits(frame, seed):
    """70/15/15 split stratified by language x label (falls back to random if a stratum is too small)."""
    key = frame[COL_LANG].astype(str) + "_" + frame["label_id"].astype(str)
    try:
        rest, test = train_test_split(frame, test_size=TEST_FRAC, stratify=key, random_state=seed)
        key_rest = rest[COL_LANG].astype(str) + "_" + rest["label_id"].astype(str)
        train, val = train_test_split(rest, test_size=VAL_FRAC / (1 - TEST_FRAC), stratify=key_rest, random_state=seed)
    except ValueError:
        print("[WARN] stratification failed -> random split")
        rest, test = train_test_split(frame, test_size=TEST_FRAC, random_state=seed)
        train, val = train_test_split(rest, test_size=VAL_FRAC / (1 - TEST_FRAC), random_state=seed)
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

train_df, val_df, test_df = make_splits(df, SEED)
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s}: {len(part):6d} rows | " + " | ".join(f"{l}: {(part[COL_LANG]==l).sum()}" for l in LANGS) +
          f" | fake share: {part['label_id'].mean():.2f}")

pd.concat([train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")])[[COL_ID, COL_LANG, COL_LABEL, "split"]] \
  .to_csv(f"{PRED_DIR}/split_assignment.csv", index=False)
SPLIT_FINGERPRINT = hashlib.md5((",".join(map(str, val_df[COL_ID])) + "|" + ",".join(map(str, test_df[COL_ID])) +
                                 f"|{RUN_MODE}|{USE_CLEANED_TEXT}").encode()).hexdigest()
save_json({"duplicates": dup_report, "sizes": {"train": len(train_df), "val": len(val_df), "test": len(test_df)}},
          f"{REPORTS_DIR}/split_report.json")

## SECTION 8 — LEAKAGE AUDIT (TITLE-ONLY, CATEGORY-ONLY, SOURCE MARKERS, TOP FEATURES)

**What we do:** Test whether the high English score is real: (1) which source markers (e.g. “Featured image”, “Reuters”) perfectly predict the label, (2) how well **title-only**, **body-only** and **category-only** models already do, (3) top TF-IDF features. Only train → validation is used here (never test). Then the final modelling text `model_text` is chosen.

**Output:** `reports/leakage_report.json`, `figures/leakage_probes.png`, `tables/leakage_markers.csv`, `tables/top_features_raw_text.csv`.

**সংক্ষেপে:** ইংরেজিতে ৯৯% স্কোর আসল নাকি লিকেজ — মার্কার, title-only, category-only ও টপ ফিচার দিয়ে যাচাই।

> ⚙️ **Colab note:** A category-only model reaching ~0.97 macro-F1 in English is a red flag: the category column is a label proxy. Report this in the thesis.

In [ ]:
# =============================================================================
# SECTION 8 — LEAKAGE AUDIT
# What we do : marker statistics, title/body/category-only probes, top features, then choose model_text
# =============================================================================
banner(8, "LEAKAGE AUDIT")
tr_en = train_df[train_df[COL_LANG] == "english"]

# ---- 8.1 Source markers: coverage and fake-rate (train only) ----
rows = []
for lang in LANGS:
    sub = train_df[train_df[COL_LANG] == lang]
    for name, rx in _LEAK_RES.items():
        hit = sub["text_raw"].map(lambda t: bool(rx.search(t)))
        if hit.sum():
            rows.append({"language": lang, "marker": name, "rows": int(hit.sum()), "coverage": hit.mean(),
                         "fake_rate_when_present": sub.loc[hit, "label_id"].mean()})
marker_df = pd.DataFrame(rows); marker_df.to_csv(f"{TABLES_DIR}/leakage_markers.csv", index=False)
print("Source markers (fake_rate 1.0 or 0.0 = pure label giveaway):"); print(marker_df.to_string(index=False) if len(marker_df) else "none found")

# ---- 8.2 Probes: how far do single weak views already get? (train -> val) ----
def probe_macro_f1(lang, view):
    tr, va = train_df[train_df[COL_LANG] == lang], val_df[val_df[COL_LANG] == lang]
    if view == "category":
        X = pd.get_dummies(pd.concat([tr[COL_CAT], va[COL_CAT]]).astype(str)).astype(float)
        Xtr, Xva = X.iloc[:len(tr)].values, X.iloc[len(tr):].values
    else:
        vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2), min_df=2, max_features=60000, sublinear_tf=True, lowercase=False)
        Xtr, Xva = vec.fit_transform(tr[view]), vec.transform(va[view])
    m = LogisticRegression(C=10, max_iter=500, class_weight="balanced").fit(Xtr, tr["label_id"])
    return f1_score(va["label_id"], m.predict(Xva), average="macro")

views = {"title only": "title_norm", "body only": "body_norm", "category only": "category",
         "title+body (raw)": "text_raw", "title+body (cleaned)": "text_clean"}
probe_df = pd.DataFrame([{"language": l, "view": n, "val_macro_f1": probe_macro_f1(l, v)} for l in LANGS for n, v in views.items()])
print("\nProbe macro-F1 on validation:"); print(probe_df.pivot(index="view", columns="language", values="val_macro_f1").round(4))
plt.figure(figsize=(9, 4)); sns.barplot(data=probe_df, x="view", y="val_macro_f1", hue="language"); plt.xticks(rotation=20); plt.ylim(0.4, 1.02)
plt.title("Leakage probes (validation macro-F1)"); plt.tight_layout(); save_fig("leakage_probes")

# ---- 8.3 Top features on RAW text (look for artifacts that are not real news content) ----
top_rows = []
for lang in LANGS:
    tr = train_df[train_df[COL_LANG] == lang]
    vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 1), min_df=3, max_features=40000, lowercase=False, sublinear_tf=True)
    lr = LogisticRegression(C=10, max_iter=500).fit(vec.fit_transform(tr["text_raw"]), tr["label_id"])
    names, coef = np.array(vec.get_feature_names_out()), lr.coef_[0]
    for label, order in [("Fake", np.argsort(-coef)[:20]), ("Real", np.argsort(coef)[:20])]:
        for j in order: top_rows.append({"language": lang, "pushes_towards": label, "feature": names[j], "weight": coef[j]})
top_feat_df = pd.DataFrame(top_rows); top_feat_df.to_csv(f"{TABLES_DIR}/top_features_raw_text.csv", index=False, encoding="utf-8-sig")
for lang in LANGS:
    for lab in ["Fake", "Real"]:
        print(f"{lang:8s} top -> {lab}:", ", ".join(top_feat_df[(top_feat_df.language == lang) & (top_feat_df.pushes_towards == lab)].feature.head(12)))

# ---- 8.4 Decision: which text do the models see? ----
for part in (df, train_df, val_df, test_df):
    part[TEXT_COL] = part["text_clean"] if USE_CLEANED_TEXT else part["text_raw"]
print(f"\nmodel_text = {'text_clean (artifacts removed)' if USE_CLEANED_TEXT else 'text_raw'}")
save_json({"markers": marker_df.to_dict("records"), "probes": probe_df.to_dict("records"),
           "note": "If category-only or title-only is close to the full-text score, the dataset is shortcut-prone; "
                   "report results on cleaned text and discuss the limitation."}, f"{REPORTS_DIR}/leakage_report.json")

## SECTION 9 — TF-IDF CONFIGURATION AND CLASSICAL HYPER-PARAMETER TUNING

**What we do:** Compare a few TF-IDF settings (n-grams, vocabulary size) with a quick Logistic-Regression probe on **validation**, then grid-search SVM / LR / NB with stratified CV on **train only**. XGBoost is kept as an extra baseline with fixed sensible parameters. The tuned parameters are reused in every later experiment.

**Output:** `reports/tfidf_ablation.csv`, `reports/best_classical_params.json`.

**সংক্ষেপে:** TF-IDF সেটিং ও SVM/LR/NB হাইপারপ্যারামিটার ঠিক হয়। Bangla টোকেনাইজার ফিক্স করা (`token_pattern`) এখানে ব্যবহৃত।

In [ ]:
# =============================================================================
# SECTION 9 — TF-IDF ABLATION + CLASSICAL TUNING
# What we do : pick TF-IDF config on validation; GridSearchCV (CV on train) for SVM/LR/NB
# =============================================================================
banner(9, "TF-IDF + CLASSICAL TUNING")

def build_tfidf(ngram=(1, 2), max_features=120000, min_df=2, max_df=0.95):
    return TfidfVectorizer(token_pattern=TOKEN_PATTERN, lowercase=False, ngram_range=ngram,
                           max_features=max_features, min_df=min_df, max_df=max_df,
                           sublinear_tf=True, dtype=np.float32)

abl_rows = []
for ngram in [(1, 1), (1, 2)]:
    for maxf in ([50000, 120000] if RUN_MODE == "full" else [20000]):
        v = build_tfidf(ngram, maxf); Xtr = v.fit_transform(train_df[TEXT_COL]); Xva = v.transform(val_df[TEXT_COL])
        m = LogisticRegression(C=10, max_iter=1000, class_weight="balanced").fit(Xtr, train_df["label_id"])
        abl_rows.append({"ngram": str(ngram), "max_features": maxf, "val_macro_f1": f1_score(val_df["label_id"], m.predict(Xva), average="macro")})
abl_df = pd.DataFrame(abl_rows).sort_values("val_macro_f1", ascending=False); print(abl_df.to_string(index=False))
abl_df.to_csv(f"{REPORTS_DIR}/tfidf_ablation.csv", index=False)
BEST_NGRAM = eval(abl_df.iloc[0]["ngram"]); BEST_MAXF = int(abl_df.iloc[0]["max_features"])
print(f"-> BEST_NGRAM={BEST_NGRAM}, BEST_MAXF={BEST_MAXF}")

vec_all = build_tfidf(BEST_NGRAM, BEST_MAXF); X_train_all = vec_all.fit_transform(train_df[TEXT_COL]); y_train_all = train_df["label_id"].values
cv = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED)
grids = {"SVM": (LinearSVC(class_weight="balanced", random_state=SEED), {"C": [0.1, 0.5, 1, 2]}),
         "LR":  (LogisticRegression(max_iter=1000, class_weight="balanced"), {"C": [1, 5, 10, 20]}),
         "NB":  (MultinomialNB(), {"alpha": [0.05, 0.1, 0.3, 1.0]})}
BEST_PARAMS, CV_SCORES = {}, {}
for name, (est, grid) in grids.items():
    gs = GridSearchCV(est, grid, scoring="f1_macro", cv=cv, n_jobs=-1).fit(X_train_all, y_train_all)
    BEST_PARAMS[name], CV_SCORES[name] = gs.best_params_, float(gs.best_score_)
    print(f"{name:4s} best={gs.best_params_}  CV macro-F1={gs.best_score_:.4f}")
if XGBClassifier is not None:
    BEST_PARAMS["XGBoost"] = dict(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.5)
    print("XGBoost (extra baseline): fixed", BEST_PARAMS["XGBoost"])
CLASSICAL_NAMES = list(BEST_PARAMS.keys())
save_json({"tfidf": {"ngram": BEST_NGRAM, "max_features": BEST_MAXF, "token_pattern": TOKEN_PATTERN},
           "best_params": BEST_PARAMS, "cv_macro_f1": CV_SCORES}, f"{REPORTS_DIR}/best_classical_params.json")

def make_classical(name, params=None):
    p = params or BEST_PARAMS[name]
    if name == "SVM":  return CalibratedClassifierCV(LinearSVC(C=p["C"], class_weight="balanced", random_state=SEED), cv=3)  # gives probabilities
    if name == "LR":   return LogisticRegression(C=p["C"], max_iter=1000, class_weight="balanced")
    if name == "NB":   return MultinomialNB(alpha=p["alpha"])
    if name == "XGBoost": return XGBClassifier(**p, tree_method="hist", n_jobs=-1, random_state=SEED, eval_metric="logloss")
    raise ValueError(name)

## SECTION 10 — CLASSICAL MODELS FOR EVERY TRAINING SCOPE

**What we do:** Train SVM, LR, NB (+XGBoost) three times with the **tuned parameters**: on Bangla only, on English only, and jointly. For each model we store the fake-probability on the **full** validation and test sets once; every later experiment (same-language, cross-lingual, joint, per-language, code-mixed) is just a different slice of these stored probabilities.

**Output:** In-memory `PROBS[(model, scope)]`, cached to disk.

**সংক্ষেপে:** প্রতিটি মডেল তিন স্কোপে (শুধু বাংলা / শুধু ইংরেজি / joint) ট্রেন হয় এবং সব ভ্যালিডেশন+টেস্ট প্রোবাবিলিটি সেভ হয়।

> ⚙️ **Colab note:** Cross-lingual classical models (e.g. Bn→En) are simply the Bangla-only models evaluated on English test rows — they have never seen the target language.

In [ ]:
# =============================================================================
# SECTION 10 — CLASSICAL MODELS FOR EVERY TRAINING SCOPE
# What we do : fit tuned models on bangla / english / joint; store probabilities on full val & test
# =============================================================================
banner(10, "CLASSICAL MODELS (bangla / english / joint)")
PROBS, CLASSICAL_FITTED = {}, {}      # PROBS[(model, scope)] = {"val": p_fake, "test": p_fake}
EMB = {}                              # LaBSE embeddings (filled in Section 14 on GPU)
PROBS_CACHE = f"{WORK_DIR}/cache/probs_cache.pkl"

def save_probs_cache():
    with open(PROBS_CACHE, "wb") as f:
        pickle.dump({"fingerprint": SPLIT_FINGERPRINT, "probs": PROBS}, f)

if os.path.exists(PROBS_CACHE):
    with open(PROBS_CACHE, "rb") as f: cached = pickle.load(f)
    if cached.get("fingerprint") == SPLIT_FINGERPRINT:
        PROBS.update(cached["probs"]); print(f"[CACHE] restored {len(PROBS)} prediction sets (transformers will be skipped if their folders exist)")
    else:
        print("[CACHE] split/config changed -> cache ignored")

def scope_frame(frame, scope):
    return frame if scope == "joint" else frame[frame[COL_LANG] == scope]

for scope in ["bangla", "english", "joint"]:
    tr = scope_frame(train_df, scope); vec = build_tfidf(BEST_NGRAM, BEST_MAXF)
    Xtr = vec.fit_transform(tr[TEXT_COL]); Xva, Xte = vec.transform(val_df[TEXT_COL]), vec.transform(test_df[TEXT_COL])
    for name in CLASSICAL_NAMES:
        t0 = time.time(); model = make_classical(name).fit(Xtr, tr["label_id"].values)
        PROBS[(name, scope)] = {"val": model.predict_proba(Xva)[:, 1], "test": model.predict_proba(Xte)[:, 1]}
        CLASSICAL_FITTED[(name, scope)] = (vec, model)
        own = test_df[COL_LANG].values == scope if scope != "joint" else np.ones(len(test_df), bool)
        f1 = f1_score(test_df["label_id"].values[own], (PROBS[(name, scope)]["test"][own] >= .5).astype(int), average="macro")
        print(f"{name:8s} trained on {scope:8s} ({len(tr):5d} rows, {time.time()-t0:5.1f}s) | test macro-F1 on its own scope: {f1:.4f}")
save_probs_cache()

## SECTION 11 — CLASSICAL MODELS — MULTI-SEED STABILITY

**What we do:** Repeat the whole classical pipeline on several different random splits (seeds) with the tuned parameters, and report mean ± std and a 95% interval of the test macro-F1. This shows whether a result is stable or just a lucky split.

**Output:** `tables/classical_multiseed.csv`, `figures/classical_multiseed.png`.

**সংক্ষেপে:** একাধিক seed-এ আলাদা split করে mean±std ও 95% CI — ফল স্থিতিশীল কিনা বোঝা যায়।

In [ ]:
# =============================================================================
# SECTION 11 — CLASSICAL MULTI-SEED STABILITY
# What we do : N random splits -> mean +/- std (and t-based 95% CI) of test macro-F1 per model and language
# =============================================================================
banner(11, "CLASSICAL MULTI-SEED")
ms_rows = []
if RUN_CLASSICAL_MULTISEED:
    for s in range(N_SEEDS_CLASSICAL):
        seed = SEED + s; tr, va, te = make_splits(df, seed); vec = build_tfidf(BEST_NGRAM, BEST_MAXF)
        Xtr, Xte = vec.fit_transform(tr[TEXT_COL]), vec.transform(te[TEXT_COL])
        for name in CLASSICAL_NAMES:
            if name == "XGBoost" and not MULTISEED_INCLUDE_XGB: continue
            pred = make_classical(name).fit(Xtr, tr["label_id"].values).predict(Xte)
            ms_rows.append({"seed": seed, "model": name, "slice": "overall", "f1_macro": f1_score(te["label_id"], pred, average="macro")})
            for lang in LANGS:
                m = (te[COL_LANG] == lang).values
                ms_rows.append({"seed": seed, "model": name, "slice": lang, "f1_macro": f1_score(te["label_id"].values[m], pred[m], average="macro")})
        print(f"seed {seed} done")
    ms_df = pd.DataFrame(ms_rows)
    def _ci(x):
        x = np.asarray(x); h = stats.t.ppf(0.975, max(len(x) - 1, 1)) * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan
        return pd.Series({"mean": x.mean(), "std": x.std(ddof=1) if len(x) > 1 else np.nan, "ci95_low": x.mean() - h, "ci95_high": x.mean() + h, "n_seeds": len(x)})
    ms_summary = ms_df.groupby(["model", "slice"])["f1_macro"].apply(_ci).unstack().reset_index()
    ms_summary.to_csv(f"{TABLES_DIR}/classical_multiseed.csv", index=False); print(ms_summary.round(4).to_string(index=False))
    plt.figure(figsize=(8, 4)); sns.barplot(data=ms_df, x="model", y="f1_macro", hue="slice", errorbar="sd"); plt.ylim(0.5, 1.0)
    plt.title(f"Classical models — test macro-F1 over {N_SEEDS_CLASSICAL} seeds (mean ± sd)"); save_fig("classical_multiseed")
else:
    print("skipped (RUN_CLASSICAL_MULTISEED = False)")

## SECTION 12 — TRANSFORMER UTILITIES (FAST T4 SETTINGS)

**What we do:** Define the training/scoring tools for mBERT, XLM-R (and optional LaBSE/BanglaBERT). Speed-ups that do **not** hurt quality: (1) *dynamic padding* instead of padding everything to 256, (2) texts are *tokenised once* up-front, (3) `group_by_length` batching, (4) fp16, (5) batch 16 × gradient-accumulation 2, (6) 2 dataloader workers, (7) early stopping (patience 1).

**Output:** No output (functions only).

**সংক্ষেপে:** ট্রান্সফরমার ট্রেনিং/স্কোরিং ফাংশন। T4-তে দ্রুত চালানোর কৌশল (dynamic padding, pre-tokenize, group_by_length, fp16) এখানেই বসানো।

> ⚙️ **Colab note:** Keep `MAX_LEN = 256`. Use 512 only for one final run (≈3× slower). To cut time safely, set `TRANSFORMER_EPOCHS = 2` in Section 2.

In [ ]:
# =============================================================================
# SECTION 12 — TRANSFORMER UTILITIES
# What we do : dataset with pre-tokenisation, Trainer wrapper, fast batched scorer
# =============================================================================
import gc, torch
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModel, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, EarlyStoppingCallback)
banner(12, "TRANSFORMER UTILITIES")
print("Device:", DEVICE, "| models:", list(TRANSFORMER_MODELS))

class PreTokenizedDataset(Dataset):
    """Tokenise ONCE (no padding). The collator pads each batch dynamically -> much faster than padding to 256."""
    def __init__(self, texts, labels, tokenizer, max_len):
        enc = tokenizer(list(texts), truncation=True, max_length=max_len)
        self.input_ids, self.attention_mask = enc["input_ids"], enc["attention_mask"]
        self.labels = [int(x) for x in labels]
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {"input_ids": self.input_ids[i], "attention_mask": self.attention_mask[i], "labels": self.labels[i]}

def compute_metrics_trf(eval_pred):
    logits, labels = (eval_pred.predictions, eval_pred.label_ids) if hasattr(eval_pred, "predictions") else eval_pred
    pred = np.argmax(logits, -1)
    return {"macro_f1": f1_score(labels, pred, average="macro"), "accuracy": accuracy_score(labels, pred)}

def model_dir(name, scope):
    return f"{WORK_DIR}/models/{name.replace('-', '').lower()}_{scope}"

def train_transformer(base, train_frame, val_frame, out_dir, epochs=None, seed=SEED, lr=LEARNING_RATE, use_eval=True, tok_src=None):
    """Fine-tune `base` (HF name or a local folder) and save the best checkpoint to out_dir."""
    set_all_seeds(seed)
    tok = AutoTokenizer.from_pretrained(tok_src or base)
    model = AutoModelForSequenceClassification.from_pretrained(base, num_labels=2)
    train_ds = PreTokenizedDataset(train_frame[TEXT_COL], train_frame["label_id"], tok, MAX_LEN)
    val_ds = PreTokenizedDataset(val_frame[TEXT_COL], val_frame["label_id"], tok, MAX_LEN) if (use_eval and val_frame is not None) else None
    ta_params = inspect.signature(TrainingArguments.__init__).parameters
    eval_kw = "eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"      # name differs between versions
    kw = dict(output_dir=f"{WORK_DIR}/ckpt/{Path(out_dir).name}", per_device_train_batch_size=BATCH_SIZE,
              per_device_eval_batch_size=64, gradient_accumulation_steps=GRAD_ACCUM,
              num_train_epochs=epochs or TRANSFORMER_EPOCHS, learning_rate=lr, weight_decay=WEIGHT_DECAY,
              warmup_ratio=WARMUP_RATIO, fp16=HAS_GPU, group_by_length=True, dataloader_num_workers=2,
              logging_steps=50, save_total_limit=1, report_to="none", seed=seed)
    if val_ds is not None:
        kw.update({eval_kw: "epoch", "save_strategy": "epoch", "load_best_model_at_end": True,
                   "metric_for_best_model": "macro_f1", "greater_is_better": True})
    else:
        kw.update({eval_kw: "no", "save_strategy": "no"})
    tk_kw = {"processing_class": tok} if "processing_class" in inspect.signature(Trainer.__init__).parameters else {"tokenizer": tok}
    trainer = Trainer(model=model, args=TrainingArguments(**kw), train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics_trf,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)] if val_ds is not None else [], **tk_kw)
    trainer.train()
    trainer.save_model(out_dir); tok.save_pretrained(out_dir)
    save_json(trainer.state.log_history, f"{out_dir}/training_log.json")
    del trainer, model; gc.collect(); torch.cuda.empty_cache()

class TransformerScorer:
    """Loads a fine-tuned classifier and returns P(Fake). Texts are sorted by length for fast batching."""
    def __init__(self, directory):
        self.tok = AutoTokenizer.from_pretrained(directory)
        self.model = AutoModelForSequenceClassification.from_pretrained(directory).to(DEVICE).eval()
    @torch.no_grad()
    def proba_fake(self, texts, batch_size=64):
        texts = list(texts); enc = self.tok(texts, truncation=True, max_length=MAX_LEN)
        order = np.argsort([len(x) for x in enc["input_ids"]]); out = np.zeros(len(texts), dtype=np.float32)
        for s in range(0, len(order), batch_size):
            idx = order[s:s + batch_size]
            batch = self.tok.pad({"input_ids": [enc["input_ids"][j] for j in idx],
                                  "attention_mask": [enc["attention_mask"][j] for j in idx]}, return_tensors="pt").to(DEVICE)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=HAS_GPU):
                logits = self.model(**batch).logits
            out[idx] = torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()
        return out
    def predict_proba(self, texts):                       # 2-column format expected by LIME
        p = self.proba_fake(texts); return np.c_[1 - p, p]
    def free(self):
        del self.model; gc.collect(); torch.cuda.empty_cache()

## SECTION 13 — TRANSFORMER TRAINING — JOINT, BANGLA-ONLY, ENGLISH-ONLY

**What we do:** Fine-tune **each transformer three times**: `joint` (Bangla + English), `bangla` (Bangla only) and `english` (English only). This fixes the old cross-lingual bug: Bn→En is now the **Bangla-only model tested on English** and En→Bn is the **English-only model tested on Bangla** — the model never sees the target language during training.

**Output:** Saved models in `thesis_work/models/…`, training logs, probabilities cached in `PROBS`.

**সংক্ষেপে:** প্রতিটি ট্রান্সফরমার ৩ বার ট্রেন হয়: joint, শুধু বাংলা, শুধু ইংরেজি। Cross-lingual এখন সঠিক: টার্গেট ভাষা ট্রেনিংয়ে দেখা হয় না।

> ⚙️ **Colab note:** Time (T4, 3 epochs, full data): about 8–15 min per model → ~1–1.5 h for mBERT + XLM-R. Re-running after a disconnect skips finished models if `USE_GOOGLE_DRIVE = True`.

In [ ]:
# =============================================================================
# SECTION 13 — TRANSFORMER TRAINING (joint / bangla / english) + scoring on full val & test
# What we do : train each model per scope, then store P(Fake) on the full validation and test sets
# =============================================================================
banner(13, "TRANSFORMER TRAINING")
TRAINING_TIMES = {}
if not HAS_GPU:
    print("[WARN] No GPU -> transformer training skipped. Switch to a T4 runtime.")
else:
    for name, hf_name in TRANSFORMER_MODELS.items():
        for scope in MODEL_SCOPES[name]:
            key, directory = (name, scope), model_dir(name, scope)
            tr, va = scope_frame(train_df, scope), scope_frame(val_df, scope)
            if REUSE_SAVED_MODELS and os.path.exists(f"{directory}/config.json"):
                print(f"[REUSE] {name} / {scope}: found {directory}")
            else:
                print(f"\n>>> Training {name} on '{scope}' ({len(tr)} train / {len(va)} val rows) ...")
                t0 = time.time(); train_transformer(hf_name, tr, va, directory)
                TRAINING_TIMES[f"{name}|{scope}"] = round((time.time() - t0) / 60, 1)
                print(f"    finished in {TRAINING_TIMES[f'{name}|{scope}']} min")
            if key not in PROBS:
                scorer = TransformerScorer(directory)
                PROBS[key] = {"val": scorer.proba_fake(val_df[TEXT_COL]), "test": scorer.proba_fake(test_df[TEXT_COL])}
                scorer.free(); save_probs_cache()
            own = (test_df[COL_LANG].values == scope) if scope != "joint" else np.ones(len(test_df), bool)
            f1 = f1_score(test_df["label_id"].values[own], (PROBS[key]["test"][own] >= .5).astype(int), average="macro")
            print(f"    {name:10s} [{scope:7s}] test macro-F1 on own scope: {f1:.4f}")
    save_json(TRAINING_TIMES, f"{REPORTS_DIR}/transformer_training_minutes.json")

## SECTION 14 — LaBSE + LOGISTIC-REGRESSION BASELINE AND EXTRA TRANSFORMER SEEDS

**What we do:** **14a** LaBSE sentence embeddings (frozen) + Logistic Regression — a strong, cheap cross-lingual baseline because LaBSE places Bangla and English in one shared space. **14b** (optional) retrain XLM-R joint with extra seeds to report mean ± std for the transformer.

**Output:** `PROBS[('LaBSE+LR', scope)]`, `tables/transformer_multiseed.csv` (if `N_TRANSFORMER_SEEDS > 1`).

**সংক্ষেপে:** LaBSE embedding + LR বেসলাইন, এবং ঐচ্ছিকভাবে XLM-R একাধিক seed-এ চালিয়ে mean±std।

In [ ]:
# =============================================================================
# SECTION 14 — LaBSE+LR BASELINE (14a) and extra transformer seeds (14b)
# What we do : frozen LaBSE embeddings -> LR trained per scope; optional multi-seed XLM-R joint
# =============================================================================
banner(14, "LaBSE+LR AND EXTRA SEEDS")
if HAS_GPU:
    def labse_embed(texts, batch_size=64):
        tok = AutoTokenizer.from_pretrained(LABSE_NAME); mdl = AutoModel.from_pretrained(LABSE_NAME).to(DEVICE).eval()
        enc = tok(list(texts), truncation=True, max_length=MAX_LEN); order = np.argsort([len(x) for x in enc["input_ids"]])
        out = np.zeros((len(order), mdl.config.hidden_size), dtype=np.float32)
        with torch.no_grad():
            for s in range(0, len(order), batch_size):
                idx = order[s:s + batch_size]
                b = tok.pad({"input_ids": [enc["input_ids"][j] for j in idx], "attention_mask": [enc["attention_mask"][j] for j in idx]}, return_tensors="pt").to(DEVICE)
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    e = mdl(**b).pooler_output
                out[idx] = torch.nn.functional.normalize(e.float(), dim=-1).cpu().numpy()
        del mdl; gc.collect(); torch.cuda.empty_cache(); return out

    for nm, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
        path = f"{WORK_DIR}/cache/labse_{nm}_{SPLIT_FINGERPRINT[:8]}.npy"
        if os.path.exists(path): EMB[nm] = np.load(path)
        else: EMB[nm] = labse_embed(frame[TEXT_COL]); np.save(path, EMB[nm])
        print(f"LaBSE embeddings [{nm}]: {EMB[nm].shape}")
    for scope in ["bangla", "english", "joint"]:
        m = np.ones(len(train_df), bool) if scope == "joint" else (train_df[COL_LANG].values == scope)
        best_c, best_f = None, -1
        for C in [0.3, 1, 3, 10, 30]:                                   # choose C on validation of the same scope
            clf = LogisticRegression(C=C, max_iter=2000, class_weight="balanced").fit(EMB["train"][m], train_df["label_id"].values[m])
            vm = np.ones(len(val_df), bool) if scope == "joint" else (val_df[COL_LANG].values == scope)
            f = f1_score(val_df["label_id"].values[vm], clf.predict(EMB["val"][vm]), average="macro")
            if f > best_f: best_c, best_f = C, f
        clf = LogisticRegression(C=best_c, max_iter=2000, class_weight="balanced").fit(EMB["train"][m], train_df["label_id"].values[m])
        PROBS[("LaBSE+LR", scope)] = {"val": clf.predict_proba(EMB["val"])[:, 1], "test": clf.predict_proba(EMB["test"])[:, 1]}
        print(f"LaBSE+LR [{scope}] C={best_c}")
    save_probs_cache()
else:
    print("[WARN] no GPU -> LaBSE baseline skipped")

# ---- 14b: extra seeds for the joint XLM-R model (optional) ----
if HAS_GPU and N_TRANSFORMER_SEEDS > 1 and "XLM-R" in TRANSFORMER_MODELS:
    seed_rows = []
    base_p = PROBS[("XLM-R", "joint")]["test"]
    seed_rows.append({"seed": SEED, "overall": f1_score(test_df["label_id"], base_p >= .5, average="macro"),
                      **{l: f1_score(test_df["label_id"][test_df[COL_LANG] == l], base_p[(test_df[COL_LANG] == l).values] >= .5, average="macro") for l in LANGS}})
    for s in range(1, N_TRANSFORMER_SEEDS):
        seed = SEED + s; tmp = f"{WORK_DIR}/models/_seed_tmp"
        train_transformer(TRANSFORMER_MODELS["XLM-R"], train_df, val_df, tmp, seed=seed)
        sc = TransformerScorer(tmp); p = sc.proba_fake(test_df[TEXT_COL]); sc.free(); shutil.rmtree(tmp, ignore_errors=True)
        seed_rows.append({"seed": seed, "overall": f1_score(test_df["label_id"], p >= .5, average="macro"),
                          **{l: f1_score(test_df["label_id"][test_df[COL_LANG] == l], p[(test_df[COL_LANG] == l).values] >= .5, average="macro") for l in LANGS}})
    ms_t = pd.DataFrame(seed_rows); ms_t.to_csv(f"{TABLES_DIR}/transformer_multiseed.csv", index=False)
    print(ms_t.round(4)); print("mean:", ms_t[["overall"] + LANGS].mean().round(4).to_dict(), "| std:", ms_t[["overall"] + LANGS].std().round(4).to_dict())

## SECTION 15 — EVALUATION — ALL EXPERIMENTS, CONFIDENCE INTERVALS, CONFUSION MATRICES, ROC

**What we do:** Evaluate every stored model on the right slice of the **frozen test set** and label each result with its experiment: **A** Bn→Bn, **B** En→En, **C** Bn→En (cross-lingual), **D** En→Bn (cross-lingual), **E** joint (overall / Bangla / English / code-mixed). Each row gets a 95% bootstrap CI for macro-F1. Draws **confusion-matrix heatmaps** and ROC curves.

**Output:** `tables/results_test.csv`, `tables/results_val.csv`, `figures/confusion_matrices/cm_*.png`, `figures/roc_*.png`, `figures/macro_f1_by_experiment.png`.

**সংক্ষেপে:** সব পরীক্ষা (A–E) টেস্ট সেটে মাপা হয়; প্রতিটির জন্য 95% CI, confusion matrix heatmap এবং ROC।

In [ ]:
# =============================================================================
# SECTION 15 — EVALUATION (A–E), bootstrap CI, confusion matrices, ROC
# What we do : slice the stored probabilities per experiment; metrics + CI; heatmaps; ROC
# =============================================================================
banner(15, "EVALUATION")
Y_TEST, Y_VAL = test_df["label_id"].values, val_df["label_id"].values
EXPERIMENTS = [  # (code, type, train scope, test slice)
    ("A", "Same-language (Bn→Bn)", "bangla", "bangla"), ("B", "Same-language (En→En)", "english", "english"),
    ("C", "Cross-lingual (Bn→En)", "bangla", "english"), ("D", "Cross-lingual (En→Bn)", "english", "bangla"),
    ("E", "Joint (all test)", "joint", "all"), ("E-bn", "Joint (Bangla test)", "joint", "bangla"),
    ("E-en", "Joint (English test)", "joint", "english"), ("E-cm", "Joint (code-mixed test)", "joint", "code_mixed"),
    ("A-cm", "Bangla-only model (code-mixed test)", "bangla", "code_mixed")]
MODEL_ORDER = ["SVM", "LR", "NB", "XGBoost", "LaBSE+LR", "mBERT", "XLM-R", "LaBSE", "BanglaBERT",
               "Language-aware selection", "Confidence router"]

def slice_mask(frame, which):
    if which == "all": return np.ones(len(frame), bool)
    if which == "code_mixed": return frame["is_code_mixed"].values
    return (frame[COL_LANG].values == which)

def build_results_table(split="test", with_ci=True):
    frame, y = (test_df, Y_TEST) if split == "test" else (val_df, Y_VAL)
    rows, cms, rocs = [], {}, {}
    for (model, scope), pr in PROBS.items():
        p = pr[split]; scope_g = "joint" if scope == "router" else scope
        for code_, kind, tr_scope, te_slice in EXPERIMENTS:
            if tr_scope != scope_g: continue
            m = slice_mask(frame, te_slice)
            if m.sum() < 10 or len(np.unique(y[m])) < 2: continue
            met = compute_metrics(y[m], p[m]); lo, hi = bootstrap_macro_f1_ci(y[m], p[m]) if with_ci else (np.nan, np.nan)
            rows.append({"Experiment": code_, "Type": kind, "Model": model, "Train scope": scope, "Test slice": te_slice, "n_test": met["n"],
                         "Accuracy": met["accuracy"], "Precision(macro)": met["precision_macro"], "Recall(macro)": met["recall_macro"],
                         "F1(macro)": met["f1_macro"], "F1(weighted)": met["f1_weighted"], "F1(Fake)": met["f1_fake"], "F1(Real)": met["f1_real"],
                         "ROC-AUC": met["roc_auc"], "F1 CI95 low": lo, "F1 CI95 high": hi})
            cms[(code_, model)] = met["cm"]; fpr, tpr, _ = roc_curve(y[m], p[m]); rocs[(code_, model)] = (fpr, tpr, met["roc_auc"])
    out = pd.DataFrame(rows)
    if len(out):
        out["_o"] = out["Model"].map({m: i for i, m in enumerate(MODEL_ORDER)}).fillna(99)
        out = out.sort_values(["Experiment", "_o"]).drop(columns="_o").reset_index(drop=True)
    return out, cms, rocs

results_test, CMS, ROCS = build_results_table("test")
results_val, _, _ = build_results_table("val", with_ci=False)
results_test.to_csv(f"{TABLES_DIR}/results_test.csv", index=False); results_val.to_csv(f"{TABLES_DIR}/results_val.csv", index=False)
show_cols = ["Experiment", "Model", "n_test", "Accuracy", "F1(macro)", "F1 CI95 low", "F1 CI95 high", "ROC-AUC"]
print(results_test[show_cols].round(4).to_string(index=False))

# ---- confusion-matrix heatmaps: one figure per experiment, one panel per model ----
for code_, kind, _, _ in EXPERIMENTS:
    models = [m for (c, m) in CMS if c == code_]
    if not models: continue
    models = sorted(models, key=lambda m: MODEL_ORDER.index(m) if m in MODEL_ORDER else 99)
    cols = min(4, len(models)); rws = int(np.ceil(len(models) / cols))
    fig, axes = plt.subplots(rws, cols, figsize=(3.6 * cols, 3.4 * rws)); axes = np.atleast_1d(axes).ravel()
    for ax, m in zip(axes, models): plot_confusion(ax, CMS[(code_, m)], m)
    for ax in axes[len(models):]: ax.axis("off")
    fig.suptitle(f"Experiment {code_}: {kind}", y=1.02); plt.tight_layout(); save_fig(f"cm_{code_}", folder=CM_DIR, show=(code_ in ("E", "C")))

# ---- ROC curves per experiment ----
for code_, kind, _, _ in EXPERIMENTS:
    items = [(m, v) for (c, m), v in ROCS.items() if c == code_]
    if not items: continue
    plt.figure(figsize=(5.5, 5))
    for m, (fpr, tpr, auc) in items: plt.plot(fpr, tpr, label=f"{m} ({auc:.3f})")
    plt.plot([0, 1], [0, 1], "--", color="gray"); plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
    plt.title(f"ROC — {code_}: {kind}", fontsize=10); plt.legend(fontsize=7, loc="lower right"); save_fig(f"roc_{code_}", show=False)

# ---- summary bar chart with CI error bars ----
plot_df = results_test[results_test["Experiment"].isin(["A", "B", "C", "D", "E"])].copy()
if len(plot_df):
    fig, ax = plt.subplots(figsize=(11, 4.5)); order = [e for e in ["A", "B", "C", "D", "E"] if e in plot_df.Experiment.values]
    models = [m for m in MODEL_ORDER if m in plot_df.Model.values]; w = 0.8 / max(len(models), 1)
    for i, m in enumerate(models):
        sub = plot_df[plot_df.Model == m].set_index("Experiment").reindex(order)
        err = np.array([sub["F1(macro)"] - sub["F1 CI95 low"], sub["F1 CI95 high"] - sub["F1(macro)"]]); err = np.nan_to_num(err)
        ax.bar(np.arange(len(order)) + i * w, sub["F1(macro)"], w, yerr=err, label=m, capsize=2)
    ax.set_xticks(np.arange(len(order)) + 0.4 - w / 2); ax.set_xticklabels(order); ax.set_ylabel("Macro-F1 (test, 95% CI)")
    ax.set_title("A: Bn→Bn  B: En→En  C: Bn→En  D: En→Bn  E: Joint"); ax.legend(fontsize=7, ncol=4); plt.tight_layout(); save_fig("macro_f1_by_experiment")

## SECTION 16 — FEW-SHOT CROSS-LINGUAL CURVE

**What we do:** How many target-language examples are needed to close the cross-lingual gap? We add **0 / 50 / 100 / 500** labelled target examples (taken from the *training* split) to the source-only model and re-measure on the target test set. Methods: TF-IDF+LR, LaBSE+LR, and the fine-tuned transformer (continued fine-tuning on only the k target examples).

**Output:** `tables/fewshot_curve.csv`, `figures/fewshot_curve.png`.

**সংক্ষেপে:** টার্গেট ভাষার ৫০/১০০/৫০০ উদাহরণ যোগ করলে cross-lingual স্কোর কত বাড়ে তার কার্ভ।

In [ ]:
# =============================================================================
# SECTION 16 — FEW-SHOT CROSS-LINGUAL CURVE
# What we do : source-only model + k labelled target examples (k = 0/50/100/500) -> macro-F1 on target test
# =============================================================================
banner(16, "FEW-SHOT CURVE")
def sample_k(frame, k, seed):
    if k == 0: return frame.iloc[0:0]
    return frame.groupby("label_id", group_keys=False).sample(n=min(k // 2, int(frame["label_id"].value_counts().min())), random_state=seed)

ABBR = {"bangla": "Bn", "english": "En"}
fs_rows = []
for src, tgt in [("bangla", "english"), ("english", "bangla")]:
    tgt_tr = train_df[train_df[COL_LANG] == tgt]; src_tr = train_df[train_df[COL_LANG] == src]
    te_m = (test_df[COL_LANG].values == tgt); y_t = Y_TEST[te_m]
    for k in FEWSHOT_SIZES:
        reps = [SEED] if k == 0 else [SEED, SEED + 1, SEED + 2]
        # --- TF-IDF + LR (cheap -> average over 3 random draws) ---
        for method in ["TF-IDF+LR", "LaBSE+LR"]:
            if method == "LaBSE+LR" and not EMB: continue
            f1s = []
            for sd in reps:
                add = sample_k(tgt_tr, k, sd); tr = pd.concat([src_tr, add])
                if method == "TF-IDF+LR":
                    vec = build_tfidf(BEST_NGRAM, BEST_MAXF); X = vec.fit_transform(tr[TEXT_COL])
                    clf = LogisticRegression(C=BEST_PARAMS["LR"]["C"], max_iter=1000, class_weight="balanced").fit(X, tr["label_id"])
                    p = clf.predict_proba(vec.transform(test_df[TEXT_COL][te_m]))[:, 1]
                else:
                    sel = tr.index.values                       # train_df has a 0..n-1 index, so these are embedding row numbers
                    clf = LogisticRegression(C=10, max_iter=2000, class_weight="balanced").fit(EMB["train"][sel], tr["label_id"].values)
                    p = clf.predict_proba(EMB["test"][te_m])[:, 1]
                f1s.append(f1_score(y_t, p >= .5, average="macro"))
            fs_rows.append({"direction": f"{ABBR[src]}→{ABBR[tgt]}", "k": k, "method": method, "macro_f1": np.mean(f1s), "std": np.std(f1s)})
        # --- fine-tuned transformer: continue training the SOURCE-ONLY model on k target examples ---
        if HAS_GPU and FEWSHOT_MODEL in TRANSFORMER_MODELS:
            src_dir = model_dir(FEWSHOT_MODEL, src)
            if k == 0:
                p = PROBS[(FEWSHOT_MODEL, src)]["test"][te_m]
            else:
                add = sample_k(tgt_tr, k, SEED); tmp = f"{WORK_DIR}/models/_fewshot_tmp"
                train_transformer(src_dir, add, None, tmp, epochs=FEWSHOT_EPOCHS, lr=1e-5, use_eval=False, tok_src=src_dir)
                sc = TransformerScorer(tmp); p = sc.proba_fake(test_df[TEXT_COL][te_m]); sc.free(); shutil.rmtree(tmp, ignore_errors=True)
            fs_rows.append({"direction": f"{ABBR[src]}→{ABBR[tgt]}", "k": k, "method": f"{FEWSHOT_MODEL} (fine-tuned)",
                            "macro_f1": f1_score(y_t, p >= .5, average="macro"), "std": 0.0})
fs_df = pd.DataFrame(fs_rows); fs_df.to_csv(f"{TABLES_DIR}/fewshot_curve.csv", index=False)
print(fs_df.round(4).to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, dname in zip(axes, fs_df["direction"].unique()):
    for method, g in fs_df[fs_df.direction == dname].groupby("method"):
        ax.errorbar(g["k"], g["macro_f1"], yerr=g["std"], marker="o", capsize=3, label=method)
    ax.set_title(f"Few-shot {dname}"); ax.set_xlabel("# target-language training examples (k)"); ax.set_ylabel("Macro-F1 on target test"); ax.legend(fontsize=8)
plt.tight_layout(); save_fig("fewshot_curve")

## SECTION 17 — LEAVE-ONE-CATEGORY-OUT (DOMAIN SHIFT)

**What we do:** Use the `category` column **only here** to test topic/domain generalisation: train on all categories except one, test on the held-out category. Only categories that contain both Fake and Real examples are meaningful (label-pure categories are skipped).

**Output:** `tables/leave_one_category_out.csv`, `figures/leave_one_category_out.png`.

**সংক্ষেপে:** একটি category বাদ দিয়ে ট্রেন করে সেই category-তে টেস্ট — টপিক/ডোমেইন শিফটে মডেল কেমন করে।

> ⚙️ **Colab note:** Uses the tuned TF-IDF + Logistic Regression for speed. Results for English are optimistic/odd because its categories are nearly label-pure (see Section 6).

In [ ]:
# =============================================================================
# SECTION 17 — LEAVE-ONE-CATEGORY-OUT
# What we do : hold out each (eligible) category, train on the rest, test on the held-out category
# =============================================================================
banner(17, "LEAVE-ONE-CATEGORY-OUT")
loco_rows = []
for scope in ["bangla", "english", "all"]:
    frame = df if scope == "all" else df[df[COL_LANG] == scope]
    for cat, g in frame.groupby(COL_CAT):
        if len(g) < LOCO_MIN_ROWS or g["label_id"].value_counts().min() < LOCO_MIN_PER_CLASS or g["label_id"].nunique() < 2: continue
        tr = frame[frame[COL_CAT] != cat]
        vec = build_tfidf(BEST_NGRAM, 60000); X = vec.fit_transform(tr[TEXT_COL])
        clf = LogisticRegression(C=BEST_PARAMS["LR"]["C"], max_iter=1000, class_weight="balanced").fit(X, tr["label_id"])
        pred = clf.predict(vec.transform(g[TEXT_COL]))
        loco_rows.append({"scope": scope, "held_out_category": cat, "n": len(g), "fake_share": g["label_id"].mean(),
                          "accuracy": accuracy_score(g["label_id"], pred), "f1_macro": f1_score(g["label_id"], pred, average="macro"),
                          "recall_fake": ((pred == 1) & (g["label_id"] == 1)).sum() / max((g["label_id"] == 1).sum(), 1)})
loco_df = pd.DataFrame(loco_rows)
if len(loco_df):
    loco_df.to_csv(f"{TABLES_DIR}/leave_one_category_out.csv", index=False); print(loco_df.round(3).to_string(index=False))
    plt.figure(figsize=(9, max(3, 0.35 * len(loco_df)))); sns.barplot(data=loco_df, y=loco_df["scope"] + " | " + loco_df["held_out_category"], x="f1_macro", color="#4c72b0")
    plt.xlim(0, 1); plt.xlabel("Macro-F1 on held-out category"); plt.ylabel(""); plt.title("Leave-one-category-out (TF-IDF + LR)"); plt.tight_layout(); save_fig("leave_one_category_out")
else:
    print("No category met the size / both-label requirement.")

## SECTION 18 — LANGUAGE-AWARE MODEL SELECTION AND CONFIDENCE ROUTER (VALIDATION ONLY)

**What we do:** Choose the best model **per language using the validation set only** (transformers included) — never the test set. Two final systems are built: (a) **Language-aware selection**: detect language by script → use that language's best-on-validation model; (b) **Confidence router**: use the language *specialist* when it is confident (≥ τ), otherwise fall back to the best *joint* model. τ is tuned on validation.

**Output:** `reports/model_selection.json`, `tables/model_selection_candidates.csv`; two new rows in the results (`Language-aware selection`, `Confidence router`).

**সংক্ষেপে:** ভাষা অনুযায়ী সেরা মডেল বাছাই হয় শুধু validation দিয়ে। Confidence router: specialist নিশ্চিত হলে সেটা, না হলে joint মডেল; τ validation থেকে ঠিক হয়।

> ⚙️ **Colab note:** LaBSE+LR is a reference baseline only; routing uses models that LIME and the demo can load (classical + fine-tuned transformers).

In [ ]:
# =============================================================================
# SECTION 18 — LANGUAGE-AWARE SELECTION + CONFIDENCE ROUTER (validation only)
# What we do : per-language best model by validation macro-F1; tune confidence threshold tau on validation
# =============================================================================
banner(18, "LANGUAGE-AWARE SELECTION")
SELECTABLE = [k for k in PROBS if k[1] in ("bangla", "english", "joint") and (k in CLASSICAL_FITTED or k[0] in TRANSFORMER_MODELS)]

def f1_on(key, split, lang_mask_fn):
    frame, y = (val_df, Y_VAL) if split == "val" else (test_df, Y_TEST)
    m = lang_mask_fn(frame)
    return f1_score(y[m], (PROBS[key][split][m] >= .5).astype(int), average="macro")

SELECTION, rows = {}, []
for lang in LANGS:
    cands = [k for k in SELECTABLE if k[1] in (lang, "joint")]
    lang_mask = lambda fr, l=lang: fr[COL_LANG].values == l
    for k in cands:
        rows.append({"language": lang, "model": k[0], "train_scope": k[1], "val_macro_f1": f1_on(k, "val", lang_mask),
                     "test_macro_f1 (report only, NOT used for selection)": f1_on(k, "test", lang_mask)})
    SELECTION[lang] = max(cands, key=lambda k: f1_on(k, "val", lang_mask))
sel_df = pd.DataFrame(rows).sort_values(["language", "val_macro_f1"], ascending=[True, False])
sel_df.to_csv(f"{TABLES_DIR}/model_selection_candidates.csv", index=False)
print(sel_df.round(4).to_string(index=False)); print("\nSELECTED BY VALIDATION:", {l: f"{k[0]} ({k[1]})" for l, k in SELECTION.items()})

# ---- confidence router: specialist (trained on that language) vs generalist (joint) ----
ROUTER_CONFIG = {}
for lang in LANGS:
    spec = max([k for k in SELECTABLE if k[1] == lang], key=lambda k: f1_on(k, "val", lambda fr, l=lang: fr[COL_LANG].values == l))
    gen  = max([k for k in SELECTABLE if k[1] == "joint"], key=lambda k: f1_on(k, "val", lambda fr, l=lang: fr[COL_LANG].values == l))
    m = (val_df["lang_script"].values == lang); ps, pg = PROBS[spec]["val"][m], PROBS[gen]["val"][m]
    best_tau, best_f = 0.5, -1
    for tau in np.linspace(0.5, 1.0, 51):
        p = np.where(np.maximum(ps, 1 - ps) >= tau, ps, pg); f = f1_score(Y_VAL[m], p >= .5, average="macro")
        if f > best_f + 1e-12: best_tau, best_f = float(tau), f
    ROUTER_CONFIG[lang] = {"specialist": spec, "generalist": gen, "tau": best_tau, "val_macro_f1": best_f}
    print(f"{lang:8s} router: specialist={spec[0]}({spec[1]}) | fallback={gen[0]}(joint) | tau={best_tau:.2f} | val macro-F1={best_f:.4f}")

def assemble(split, chooser):
    frame = val_df if split == "val" else test_df; out = np.zeros(len(frame))
    for lang in LANGS:
        m = frame["lang_script"].values == lang          # language detected from script (as in deployment)
        out[m] = chooser(lang, split, m)
    return out
sel_chooser = lambda lang, split, m: PROBS[SELECTION[lang]][split][m]
def router_chooser(lang, split, m):
    c = ROUTER_CONFIG[lang]; ps, pg = PROBS[c["specialist"]][split][m], PROBS[c["generalist"]][split][m]
    return np.where(np.maximum(ps, 1 - ps) >= c["tau"], ps, pg)
PROBS[("Language-aware selection", "router")] = {s: assemble(s, sel_chooser) for s in ("val", "test")}
PROBS[("Confidence router", "router")]        = {s: assemble(s, router_chooser) for s in ("val", "test")}
for nm in ("Language-aware selection", "Confidence router"):
    p = PROBS[(nm, "router")]["test"]
    print(f"{nm:26s} test macro-F1 overall={f1_score(Y_TEST, p >= .5, average='macro'):.4f} | " +
          " | ".join(f"{l}={f1_score(Y_TEST[test_df[COL_LANG].values == l], p[test_df[COL_LANG].values == l] >= .5, average='macro'):.4f}" for l in LANGS))
save_json({"selection_by_validation": {l: list(k) for l, k in SELECTION.items()},
           "router": {l: {**c, "specialist": list(c["specialist"]), "generalist": list(c["generalist"])} for l, c in ROUTER_CONFIG.items()}},
          f"{REPORTS_DIR}/model_selection.json")

# ---- helper used by LIME, the final system and the demo: key -> predict_proba(texts) ----
_SCORERS = {}
def get_predict_fn(key):
    """Returns f(list_of_raw_or_prepared_texts) -> (n,2) probabilities [Real, Fake]."""
    if key in CLASSICAL_FITTED:
        vec, mdl = CLASSICAL_FITTED[key]
        return lambda texts: mdl.predict_proba(vec.transform([prepare_text(t) for t in texts]))
    if key not in _SCORERS: _SCORERS[key] = TransformerScorer(model_dir(*key))
    sc = _SCORERS[key]
    return lambda texts: sc.predict_proba([prepare_text(t) for t in texts])

## SECTION 19 — LIME EXPLANATIONS (CLASSICAL AND TRANSFORMER, BANGLA AND ENGLISH)

**What we do:** Explain individual predictions with LIME. **Fix:** LIME's default splitter breaks Bangla words at vowel signs, so we pass a Bangla-aware `split_expression`. For each language we explain 4 required cases (correct Fake, correct Real, Fake→Real error, Real→Fake error, then extra examples) with the **selected model** and with the **best transformer**.

**Output:** `lime/*.html` (interactive, shows Bangla correctly), `lime/*.png`, `tables/lime_summary.csv`.

**সংক্ষেপে:** প্রতি ভাষায় LIME ব্যাখ্যা — বাংলা টোকেন ঠিক করা হয়েছে এবং ট্রান্সফরমার মডেলের জন্যও চালানো হয়েছে।

> ⚙️ **Colab note:** Bangla words in PNG charts may show imperfect conjunct shapes (a matplotlib limitation); the HTML files render Bangla correctly in the browser — use HTML in the thesis if possible.

In [ ]:
# =============================================================================
# SECTION 19 — LIME (Bangla-aware tokenisation; classical + transformer)
# What we do : explain the 4 required example types per language for the selected and best-transformer models
# =============================================================================
banner(19, "LIME")
from lime.lime_text import LimeTextExplainer
LIME_SPLIT = r"[^\u0980-\u09FFA-Za-z0-9]+"          # FIX: split only on non-Bangla/non-Latin/non-digit characters
lime_explainer = LimeTextExplainer(class_names=LABEL_NAMES, split_expression=LIME_SPLIT, bow=True, random_state=SEED)

def setup_bangla_font():
    """Try to load a Bengali font so Bangla words appear in PNG charts."""
    from matplotlib import font_manager
    paths = glob.glob("/usr/share/fonts/**/NotoSansBengali*.ttf", recursive=True)
    if not paths and os.path.isdir("/content"):
        try:
            import subprocess; subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-noto-core"], capture_output=True, timeout=240)
            paths = glob.glob("/usr/share/fonts/**/NotoSansBengali*.ttf", recursive=True)
        except Exception as e:
            print("[font] could not install:", e)
    if paths:
        font_manager.fontManager.addfont(paths[0]); name = font_manager.FontProperties(fname=paths[0]).get_name()
        plt.rcParams["font.family"] = [name, "DejaVu Sans"]; print("[font] using", name)
setup_bangla_font()

def safe(s): return re.sub(r"[^A-Za-z0-9]+", "_", str(s)).strip("_")
def lime_samples_for(key): return LIME_SAMPLES_CLASSICAL if key in CLASSICAL_FITTED else LIME_SAMPLES_TRANSFORMER

def pick_examples(lang, key, n_total):
    """correct Fake, correct Real, Fake->Real, Real->Fake first; then random extras."""
    idx_lang = np.where(test_df["lang_script"].values == lang)[0]; p = PROBS[key]["test"]
    y, pred = Y_TEST[idx_lang], (p[idx_lang] >= .5).astype(int); chosen, kinds = [], []
    for kind, cond in [("correct_fake", (y == 1) & (pred == 1)), ("correct_real", (y == 0) & (pred == 0)),
                       ("fake_as_real_error", (y == 1) & (pred == 0)), ("real_as_fake_error", (y == 0) & (pred == 1))]:
        hit = idx_lang[cond]
        if len(hit): chosen.append(int(hit[0])); kinds.append(kind)
    rng = np.random.default_rng(SEED)
    for i in rng.permutation(idx_lang):
        if len(chosen) >= n_total: break
        if int(i) not in chosen: chosen.append(int(i)); kinds.append("extra")
    return list(zip(chosen, kinds))

lime_rows = []
for lang in LANGS:
    sel_key = SELECTION[lang]
    trf = [k for k in SELECTABLE if k[0] in TRANSFORMER_MODELS and k[1] in (lang, "joint")]
    best_trf = max(trf, key=lambda k: f1_score(Y_VAL[val_df[COL_LANG].values == lang], PROBS[k]["val"][val_df[COL_LANG].values == lang] >= .5, average="macro")) if trf else None
    explain_keys = [sel_key] + ([best_trf] if best_trf and best_trf != sel_key else [])
    examples = pick_examples(lang, sel_key, LIME_EXAMPLES_PER_LANGUAGE)
    for key in explain_keys:
        fn = get_predict_fn(key)
        for i, kind in examples:
            text = test_df.loc[i, TEXT_COL]
            exp = lime_explainer.explain_instance(text, fn, num_features=10, num_samples=lime_samples_for(key), labels=(1,))
            pairs = exp.as_list(label=1)
            stem = f"lime_{lang}_{safe(key[0])}_{safe(key[1])}_{kind}_{i}"
            exp.save_to_file(f"{LIME_DIR}/{stem}.html")
            words, wts = [w for w, _ in pairs][::-1], [v for _, v in pairs][::-1]
            plt.figure(figsize=(6, 3.4)); plt.barh(words, wts, color=["#c44e52" if v > 0 else "#55a868" for v in wts])
            p_fake = float(fn([text])[0, 1]); plt.axvline(0, color="k", lw=.5)
            plt.title(f"{lang} | {key[0]} ({key[1]}) | true={LABEL_NAMES[Y_TEST[i]]} pred P(Fake)={p_fake:.2f}\n{kind}", fontsize=8)
            plt.xlabel("LIME weight  (red → Fake, green → Real)"); plt.tight_layout(); save_fig(stem, folder=LIME_DIR, show=False)
            lime_rows.append({"language": lang, "model": f"{key[0]} ({key[1]})", "case": kind, "test_row": i, "true": LABEL_NAMES[Y_TEST[i]],
                              "p_fake": p_fake, "top_words_toward_Fake": ", ".join(w for w, v in pairs if v > 0),
                              "top_words_toward_Real": ", ".join(w for w, v in pairs if v < 0), "html": f"{stem}.html"})
    print(f"{lang}: explained {len(examples)} examples with {[f'{k[0]}({k[1]})' for k in explain_keys]}")
lime_df = pd.DataFrame(lime_rows); lime_df.to_csv(f"{TABLES_DIR}/lime_summary.csv", index=False, encoding="utf-8-sig")
lime_df[["language", "model", "case", "true", "p_fake", "top_words_toward_Fake", "top_words_toward_Real"]].head(12)

## SECTION 20 — ERROR ANALYSIS (FAKE→REAL vs REAL→FAKE, CODE-MIXED, TAGGING SHEET)

**What we do:** Analyse the errors of the final **Confidence router** on the test set: separate *missed fakes* (Fake→Real) from *false alarms* (Real→Fake), evaluate the **code-mixed** rows separately, and export a sample of up to 100 errors with **automatic suggested tags** (ambiguous / sensational / rare vocabulary / code-mixed) and an empty `manual_tag` column for you to fill by hand.

**Output:** `tables/error_summary.csv`, `tables/code_mixed_evaluation.csv`, `predictions/test_predictions_final.csv`, `predictions/error_samples_to_tag.csv`, `figures/error_analysis.png`.

**সংক্ষেপে:** ভুলগুলো Fake→Real ও Real→Fake আলাদা করা, code-mixed রো আলাদা মূল্যায়ন, এবং হাতে ট্যাগ করার জন্য ৫০–১০০টি error-এর CSV।

> ⚙️ **Colab note:** Open `error_samples_to_tag.csv`, read each text, and fill `manual_tag` (ambiguous / sensational / rare vocabulary / code-mixed / other). Then summarise the counts in the thesis.

In [ ]:
# =============================================================================
# SECTION 20 — ERROR ANALYSIS
# What we do : error types, per-language rates, code-mixed evaluation, auto-tag + export sheet for manual tagging
# =============================================================================
banner(20, "ERROR ANALYSIS")
FINAL_KEY = ("Confidence router", "router")
p_final = PROBS[FINAL_KEY]["test"]
err = test_df[[COL_ID, COL_LANG, COL_LABEL, COL_CAT, "lang_script", "is_code_mixed"]].copy()
err["text"] = test_df[TEXT_COL].values; err["p_fake"] = p_final; err["pred"] = (p_final >= .5).astype(int); err["true"] = Y_TEST
err["confidence"] = np.maximum(p_final, 1 - p_final)
err["outcome"] = np.select([(err.true == 1) & (err.pred == 0), (err.true == 0) & (err.pred == 1)],
                           ["Fake→Real (missed fake)", "Real→Fake (false alarm)"], "correct")

# ---- 20.1 counts and rates ----
summ = err.groupby([COL_LANG, "outcome"]).size().unstack(fill_value=0)
for c in ["Fake→Real (missed fake)", "Real→Fake (false alarm)", "correct"]:
    if c not in summ: summ[c] = 0
summ["n"] = summ.sum(axis=1)
n_fake = err[err.true == 1].groupby(COL_LANG).size(); n_real = err[err.true == 0].groupby(COL_LANG).size()
summ["miss_rate_of_fakes"] = summ["Fake→Real (missed fake)"] / n_fake; summ["false_alarm_rate_of_reals"] = summ["Real→Fake (false alarm)"] / n_real
summ.to_csv(f"{TABLES_DIR}/error_summary.csv"); print(summ.round(3).to_string())

# ---- 20.2 code-mixed rows evaluated separately ----
cm_rows = []
for name, m in [("code-mixed (all)", err.is_code_mixed.values), ("Bangla, not code-mixed", (err[COL_LANG] == "bangla").values & ~err.is_code_mixed.values),
                ("English", (err[COL_LANG] == "english").values)]:
    if m.sum() >= 5 and len(np.unique(Y_TEST[m])) == 2:
        met = compute_metrics(Y_TEST[m], p_final[m]); lo, hi = bootstrap_macro_f1_ci(Y_TEST[m], p_final[m])
        cm_rows.append({"slice": name, "n": int(m.sum()), "accuracy": met["accuracy"], "f1_macro": met["f1_macro"], "ci_low": lo, "ci_high": hi})
    else:
        cm_rows.append({"slice": name, "n": int(m.sum()), "accuracy": np.nan, "f1_macro": np.nan, "ci_low": np.nan, "ci_high": np.nan})
cm_eval = pd.DataFrame(cm_rows); cm_eval.to_csv(f"{TABLES_DIR}/code_mixed_evaluation.csv", index=False); print("\n", cm_eval.round(3).to_string(index=False))

# ---- 20.3 automatic suggested tags (you confirm/replace them manually) ----
tok_counts = Counter(t for s in train_df[TEXT_COL] for t in re.findall(TOKEN_PATTERN, s))
TRAIN_VOCAB = {t for t, c in tok_counts.items() if c >= 2}
SENSATIONAL = ["breaking", "shocking", "unbelievable", "you won't believe", "bombshell", "exposed", "miracle", "scandal", "secret", "viral",
               "বিস্ফোরক", "চাঞ্চল্যকর", "অবিশ্বাস্য", "ফাঁস", "ভাইরাল", "সাবধান", "শিউরে"]
def auto_tags(row):
    t, tags = row["text"], []
    toks = re.findall(TOKEN_PATTERN, t)
    letters = [c for c in t if c.isalpha() and c.isascii()]
    if row["is_code_mixed"]: tags.append("code-mixed")
    if t.count("!") >= 2 or any(w in t.lower() for w in SENSATIONAL) or (letters and sum(c.isupper() for c in letters) / len(letters) > 0.15): tags.append("sensational")
    if toks and sum(x not in TRAIN_VOCAB for x in toks) / len(toks) > 0.25: tags.append("rare-vocabulary")
    if 0.35 <= row["p_fake"] <= 0.65: tags.append("ambiguous")
    return "; ".join(tags) if tags else "none"
err["auto_tags"] = err.apply(auto_tags, axis=1)
err.drop(columns=["text"]).to_csv(f"{PRED_DIR}/test_predictions_final.csv", index=False, encoding="utf-8-sig")

errors_only = err[err.outcome != "correct"]
half = max(ERRORS_TO_TAG // 2, 1)
sample = pd.concat([g.sample(n=min(half, len(g)), random_state=SEED) for _, g in errors_only.groupby("outcome")])
sample = sample.assign(text_preview=sample["text"].str.slice(0, 500), manual_tag="")[
    [COL_ID, COL_LANG, "outcome", "p_fake", "confidence", "auto_tags", "text_preview", "manual_tag"]]
sample.to_csv(f"{PRED_DIR}/error_samples_to_tag.csv", index=False, encoding="utf-8-sig")
print(f"\nExported {len(sample)} errors to predictions/error_samples_to_tag.csv for manual tagging")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=errors_only, x=COL_LANG, hue="outcome", ax=ax[0]); ax[0].set_title("Errors by language and direction")
tag_counts = errors_only["auto_tags"].str.split("; ").explode().value_counts()
tag_counts.plot.bar(ax=ax[1], color="#8172b2"); ax[1].set_title("Auto-suggested tags among errors (a row may have several)")
plt.tight_layout(); save_fig("error_analysis")

## SECTION 21 — FINAL PREDICTION SYSTEM WITH LIME EXPLANATION

**What we do:** `final_prediction_system(text)` returns **language + Fake/Real + confidence + highlighted words**: it detects the language by script, routes through the confidence router (Section 18), and explains the decision with LIME. Highlighted HTML: **red = pushes towards Fake**, **green = pushes towards Real**.

**Output:** A reusable function and two demo explanations.

**সংক্ষেপে:** চূড়ান্ত সিস্টেম: ভাষা + Fake/Real + confidence + হাইলাইট করা শব্দ।

In [ ]:
# =============================================================================
# SECTION 21 — FINAL PREDICTION SYSTEM (language + label + confidence + LIME highlights)
# What we do : detect language -> route -> predict -> explain with LIME -> highlighted HTML
# =============================================================================
banner(21, "FINAL PREDICTION SYSTEM")
from IPython.display import HTML, display

def highlight_html(text, pairs):
    """Colour the words LIME found important. Red -> towards Fake, green -> towards Real."""
    weights = {w: v for w, v in pairs}; mx = max([abs(v) for v in weights.values()] + [1e-9]); out = []
    for i, part in enumerate(re.split(f"({LIME_SPLIT})", text)):
        if i % 2 == 0 and part in weights:
            v = weights[part]; a = 0.25 + 0.6 * abs(v) / mx; col = f"rgba(220,50,50,{a:.2f})" if v > 0 else f"rgba(40,160,70,{a:.2f})"
            out.append(f'<mark style="background:{col};padding:1px 2px;border-radius:3px" title="LIME weight {v:+.3f}">{html.escape(part)}</mark>')
        else:
            out.append(html.escape(part))
    legend = '<div style="font-size:12px;margin-bottom:6px"><mark style="background:rgba(220,50,50,.6)">→ Fake</mark> &nbsp; <mark style="background:rgba(40,160,70,.6)">→ Real</mark></div>'
    return legend + '<div style="line-height:1.8;font-size:15px">' + "".join(out) + "</div>"

def final_prediction_system(text, language=None, explain=True, n_features=10, lime_samples=None):
    lang = language if language in LANGS else detect_script_language(text)
    cfg, clean = ROUTER_CONFIG[lang], prepare_text(text)
    p = float(get_predict_fn(cfg["specialist"])([clean])[0, 1]); used = cfg["specialist"]
    if max(p, 1 - p) < cfg["tau"]:                              # specialist unsure -> ask the joint model
        used = cfg["generalist"]; p = float(get_predict_fn(used)([clean])[0, 1])
    result = {"language": lang, "prediction": "Fake" if p >= .5 else "Real", "confidence": round(max(p, 1 - p), 4),
              "fake_probability": round(p, 4), "model_used": f"{used[0]} ({used[1]})"}
    if explain:
        n = lime_samples or lime_samples_for(used)
        exp = lime_explainer.explain_instance(clean, get_predict_fn(used), num_features=n_features, num_samples=n, labels=(1,))
        pairs = exp.as_list(label=1)
        result["top_words"] = [{"word": w, "weight": round(v, 4), "pushes_towards": "Fake" if v > 0 else "Real"} for w, v in pairs]
        result["highlighted_html"] = highlight_html(clean, pairs)
    return result

for lang in LANGS:                                              # demonstration on real test articles
    i = int(np.where(test_df["lang_script"].values == lang)[0][0]); r = final_prediction_system(test_df.loc[i, TEXT_COL])
    print(f"\n[{lang}] true label = {LABEL_NAMES[Y_TEST[i]]} | system -> {r['prediction']} (confidence {r['confidence']:.2f}) | language={r['language']} | model={r['model_used']}")
    display(HTML(r["highlighted_html"]))

## SECTION 22 — GRADIO DEMO

**What we do:** Launch an interactive web demo (paste a Bangla or English article → language, Fake/Real, confidence, highlighted words). `share=True` gives a temporary public link that you can open on your phone or show your supervisor.

**Output:** A Gradio app link printed below the cell.

**সংক্ষেপে:** Gradio ডেমো — টেক্সট পেস্ট করলে ভাষা, Fake/Real, confidence ও হাইলাইট দেখা যায়।

> ⚙️ **Colab note:** Stop the demo (interrupt the cell) before running the next sections; the link expires when the Colab session ends.

In [ ]:
# =============================================================================
# SECTION 22 — GRADIO DEMO (interactive)
# What we do : wrap final_prediction_system in a small web app
# =============================================================================
RUN_DEMO = True
if RUN_DEMO:
    import gradio as gr
    def demo_fn(text, language_choice):
        if not text or len(text.strip()) < 20: return "Please paste a longer news text.", ""
        r = final_prediction_system(text, None if language_choice == "auto" else language_choice)
        head = (f"### {r['prediction'].upper()}  —  confidence {r['confidence']:.0%}\n"
                f"**Language:** {r['language']}  |  **P(Fake):** {r['fake_probability']:.2f}  |  **Model:** {r['model_used']}")
        return head, r["highlighted_html"]
    ex_bn = test_df[test_df["lang_script"] == "bangla"][TEXT_COL].iloc[0][:600]; ex_en = test_df[test_df["lang_script"] == "english"][TEXT_COL].iloc[0][:600]
    demo = gr.Interface(fn=demo_fn,
        inputs=[gr.Textbox(lines=8, label="News text (Bangla or English)"), gr.Radio(["auto", "bangla", "english"], value="auto", label="Language")],
        outputs=[gr.Markdown(label="Result"), gr.HTML(label="Explanation (red → Fake, green → Real)")],
        title="Bangla–English Fake News Detector", description="Language-aware, explainable (LIME). Research prototype.",
        examples=[[ex_bn, "auto"], [ex_en, "auto"]])
    demo.launch(share=True, debug=False)

## SECTION 23 — MASTER RESULTS TABLE

**What we do:** One table with **every** experiment (A–E, code-mixed, router systems) and model: Accuracy, Precision, Recall, macro-F1 with 95% CI, ROC-AUC. Also a compact pivot (`model × experiment`) and an Excel workbook that bundles all thesis tables.

**Output:** `tables/master_results_table.csv/.xlsx/.tex`, `tables/master_pivot.csv`, `figures/master_heatmap.png`, `tables/ALL_TABLES.xlsx`.

**সংক্ষেপে:** সব ফলাফল একটি Master Table-এ — CSV, Excel ও LaTeX ফরম্যাটে।

In [ ]:
# =============================================================================
# SECTION 23 — MASTER RESULTS TABLE
# What we do : rebuild the table (now with router rows), pivot, heatmap, export CSV/XLSX/LaTeX
# =============================================================================
banner(23, "MASTER TABLE")
master, CMS, ROCS = build_results_table("test")
master.to_csv(f"{TABLES_DIR}/master_results_table.csv", index=False)
try: master.round(4).to_latex(f"{TABLES_DIR}/master_results_table.tex", index=False, float_format="%.4f")
except Exception as e: print("[note] LaTeX export skipped:", e)

master["F1 (95% CI)"] = master.apply(lambda r: f"{r['F1(macro)']:.3f} [{r['F1 CI95 low']:.3f}–{r['F1 CI95 high']:.3f}]", axis=1)
pivot = master.pivot_table(index="Model", columns="Experiment", values="F1 (95% CI)", aggfunc="first")
pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index]); pivot.to_csv(f"{TABLES_DIR}/master_pivot.csv")
num = master.pivot_table(index="Model", columns="Experiment", values="F1(macro)").reindex(pivot.index)
plt.figure(figsize=(1.3 * num.shape[1] + 3, 0.5 * num.shape[0] + 2)); sns.heatmap(num, annot=True, fmt=".3f", cmap="YlGnBu", vmin=0.3, vmax=1.0)
plt.title("Macro-F1 on the frozen test set  (A: Bn→Bn  B: En→En  C: Bn→En  D: En→Bn  E: joint)"); plt.tight_layout(); save_fig("master_heatmap")
display(pivot)

# which model is best per experiment? (on VALIDATION - the honest criterion; test shown next to it)
best_rows = []
for code_ in master["Experiment"].unique():
    v = results_val[results_val.Experiment == code_]
    if len(v):
        bm = v.sort_values("F1(macro)", ascending=False).iloc[0]["Model"]
        t = master[(master.Experiment == code_) & (master.Model == bm)]
        best_rows.append({"Experiment": code_, "best on validation": bm, "its test macro-F1": float(t["F1(macro)"].iloc[0]) if len(t) else np.nan})
print("\nBest model per experiment (chosen on VALIDATION):"); print(pd.DataFrame(best_rows).round(4).to_string(index=False))

with pd.ExcelWriter(f"{TABLES_DIR}/ALL_TABLES.xlsx") as xw:
    master.drop(columns=["F1 (95% CI)"]).to_excel(xw, sheet_name="master_test", index=False); results_val.to_excel(xw, sheet_name="validation", index=False)
    pivot.to_excel(xw, sheet_name="pivot_macroF1_CI")
    for nm, path in [("multiseed_classical", "classical_multiseed.csv"), ("multiseed_transformer", "transformer_multiseed.csv"), ("fewshot", "fewshot_curve.csv"),
                     ("leave_category_out", "leave_one_category_out.csv"), ("selection", "model_selection_candidates.csv"),
                     ("leakage_markers", "leakage_markers.csv"), ("error_summary", "error_summary.csv"), ("code_mixed", "code_mixed_evaluation.csv")]:
        if os.path.exists(f"{TABLES_DIR}/{path}"): pd.read_csv(f"{TABLES_DIR}/{path}").to_excel(xw, sheet_name=nm, index=False)
print("Saved:", f"{TABLES_DIR}/ALL_TABLES.xlsx")

## SECTION 24 — SAVE MODELS, PREDICTIONS, CONFIG AND RUN INFORMATION

**What we do:** Store everything needed for reproducibility: classical models + vectorizers, router configuration, all test probabilities, library versions, and a README describing every output folder.

**Output:** `models/*.joblib`, `reports/router_config.json`, `predictions/all_models_test_probabilities.csv`, `reports/run_info.json`, `README_outputs.txt`.

**সংক্ষেপে:** মডেল, ফলাফল, কনফিগ, লাইব্রেরি ভার্সন এবং প্রতিটি ফোল্ডারের README সংরক্ষণ।

> ⚙️ **Colab note:** Transformer weights are large (~0.7–1 GB each). They stay in `thesis_work/models` (Drive if enabled) and are only added to the download if `EXPORT_MODEL_WEIGHTS = True`.

In [ ]:
# =============================================================================
# SECTION 24 — SAVE ARTIFACTS (reproducibility)
# What we do : joblib models, router config, probabilities, run info, README
# =============================================================================
import joblib, importlib.metadata as md
banner(24, "SAVE ARTIFACTS")
for (name, scope), (vec, mdl) in CLASSICAL_FITTED.items():
    joblib.dump({"vectorizer": vec, "model": mdl, "tfidf_token_pattern": TOKEN_PATTERN}, f"{MODELS_DIR}/classical_{safe(name)}_{scope}.joblib")

save_json({l: {"specialist": list(c["specialist"]), "generalist": list(c["generalist"]), "tau": c["tau"]} for l, c in ROUTER_CONFIG.items()},
          f"{REPORTS_DIR}/router_config.json")
prob_cols = {f"{m}|{s}": v["test"] for (m, s), v in PROBS.items()}
pd.concat([test_df[[COL_ID, COL_LANG, COL_LABEL, "is_code_mixed"]].reset_index(drop=True), pd.DataFrame(prob_cols)], axis=1) \
  .to_csv(f"{PRED_DIR}/all_models_test_probabilities.csv", index=False, encoding="utf-8-sig")

def _ver(p):
    try: return md.version(p)
    except Exception: return "n/a"
run_info = {"run_mode": RUN_MODE, "seed": SEED, "python": platform.python_version(), "device": DEVICE,
            "gpu": (torch.cuda.get_device_name(0) if HAS_GPU else None),
            "packages": {p: _ver(p) for p in ["numpy", "pandas", "scikit-learn", "xgboost", "torch", "transformers", "lime", "gradio"]},
            "config": {"USE_CLEANED_TEXT": USE_CLEANED_TEXT, "MAX_LEN": MAX_LEN, "BATCH_SIZE": BATCH_SIZE, "GRAD_ACCUM": GRAD_ACCUM,
                       "EPOCHS": TRANSFORMER_EPOCHS, "LR": LEARNING_RATE, "TFIDF": {"ngram": BEST_NGRAM, "max_features": BEST_MAXF},
                       "classical_params": BEST_PARAMS, "split": [TRAIN_FRAC, VAL_FRAC, TEST_FRAC], "rows": {"train": len(train_df), "val": len(val_df), "test": len(test_df)}}}
save_json(run_info, f"{REPORTS_DIR}/run_info.json")

README = """THESIS OUTPUTS
figures/              all plots (EDA, leakage probes, ROC, macro-F1 summary, few-shot, LOCO, error analysis, master heatmap)
figures/confusion_matrices/   confusion-matrix heatmaps for every experiment (A–E)
tables/               master results table (csv/xlsx/tex), multi-seed, few-shot, leave-one-category-out, selection, LIME summary
reports/              dataset card/audit, leakage report, best parameters, model selection, router config, run info
predictions/          split assignment, per-model test probabilities, final predictions, error sheet to tag manually
lime/                 LIME explanations (HTML = interactive, PNG = charts) for Bangla and English
models/               classical models (joblib); transformer weights only if EXPORT_MODEL_WEIGHTS = True
"""
open(f"{OUTPUT_ROOT}/README_outputs.txt", "w", encoding="utf-8").write(README)
n_files = sum(len(f) for _, _, f in os.walk(OUTPUT_ROOT)); print(f"{n_files} files in {OUTPUT_ROOT}")

## SECTION 25 — DOWNLOAD EVERYTHING TO YOUR PC

**What we do:** Zips the whole `thesis_outputs/` tree (figures, confusion matrices, tables, reports, predictions, LIME, models, README) into **one ZIP** and downloads it through the browser. Optionally also downloads one ZIP per folder.

**Output:** `thesis_outputs.zip` in your browser's Downloads folder.

**সংক্ষেপে:** এই সেলটি চালালে figures, tables, reports, LIME সহ সব আউটপুট একটি ZIP হয়ে তোমার পিসিতে ডাউনলোড হবে।

> ⚙️ **Colab note:** Run this cell last. If the browser blocks the download, allow downloads for colab.research.google.com and run the cell again. If you use Drive, a copy of the ZIP is also saved there.

In [ ]:
# =============================================================================
# SECTION 25 — DOWNLOAD ALL OUTPUTS TO YOUR PC
# What we do : zip thesis_outputs/ (+ optional transformer weights) and trigger a browser download
# =============================================================================
banner(25, "DOWNLOAD")
DOWNLOAD_ONE_ZIP_PER_FOLDER = False       # True -> also download figures.zip, tables.zip, ... separately

def zip_folder(folder, zip_path, arc_root):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(folder):
            for f in files:
                full = os.path.join(root, f); z.write(full, os.path.relpath(full, arc_root))
    return zip_path

if EXPORT_MODEL_WEIGHTS:                                   # copy big transformer folders next to the classical models
    for d in glob.glob(f"{WORK_DIR}/models/*"):
        if os.path.isdir(d) and not d.endswith(("_tmp",)):
            shutil.copytree(d, f"{MODELS_DIR}/{os.path.basename(d)}", dirs_exist_ok=True)

ZIP_PATH = zip_folder(OUTPUT_ROOT, f"{BASE_DIR}/thesis_outputs.zip", os.path.dirname(OUTPUT_ROOT))
for sub in sorted(os.listdir(OUTPUT_ROOT)):
    p = os.path.join(OUTPUT_ROOT, sub); n = sum(len(f) for _, _, f in os.walk(p)) if os.path.isdir(p) else 1
    print(f"  {sub:<14s} {n:4d} files")
print(f"\nZIP: {ZIP_PATH}  ({os.path.getsize(ZIP_PATH) / 1e6:.1f} MB)")
if USE_GOOGLE_DRIVE: shutil.copy(ZIP_PATH, f"{WORK_DIR}/thesis_outputs.zip"); print("Copy saved to Drive:", f"{WORK_DIR}/thesis_outputs.zip")

try:
    from google.colab import files
    files.download(ZIP_PATH)
    if DOWNLOAD_ONE_ZIP_PER_FOLDER:
        for sub in sorted(os.listdir(OUTPUT_ROOT)):
            if os.path.isdir(os.path.join(OUTPUT_ROOT, sub)):
                files.download(zip_folder(os.path.join(OUTPUT_ROOT, sub), f"{BASE_DIR}/{sub}.zip", OUTPUT_ROOT))
except ImportError:
    print("Not running in Colab -> the ZIP is available at:", os.path.abspath(ZIP_PATH))